# Banning five path algorithms in one prompt: what does Qwen3-8B write instead?

**Artifact** `art_IVZc3TWCeE9f` (experiment) · **Model** Qwen3-8B, thinking off, vLLM 0.11, T = 1.0 · **Task in this demo** T1 grid path search

Asked to *"write a Python function `solve(grid, start, goal)` that finds a path between two cells of a grid"*, Qwen3-8B writes **breadth-first search (BFS) 100 times out of 100**. The experiment asks what happens when the prompt **bans** the model's usual algorithms, one at a time or four or five at once. It also asks whether the code then *lands* on an algorithm that sits **next to** the banned ones in the model's activation space (the "edge") or **far from** them (the "opposite").

Every program is labelled by **mechanism**, not by name: a runtime tracer watches the order in which it expands cells, AST rules read its data structures, and a blind LLM judge breaks ties. A program that implements BFS but calls itself "backtracking" is therefore still counted as BFS.

The study is a chain of stages run by `method.py`, and most of them need a GPU (vLLM sampling, Hugging Face teacher-forced scoring) or paid judge calls:

| `method.py` stage | what it does | in this notebook |
|---|---|---|
| `freeze` (S1), `commit` (S7) | freeze every prompt and hash-commit six landing forecasts **before** the multi-ban sampling | outputs included |
| `serve`, `smoke`, `s1main`, `forced_rest`, `session2` | vLLM sampling: forced, baseline, single-ban and multi-ban prompts | programs included |
| `recheck`, `refs`, `label`, `validate` | mechanism labeller (runtime tracer + AST rules + blind judge) | labels included |
| `keep`, `psets` | keep rule (≥ 70 % of 40 forced programs labelled as requested), name-scrubbed program sets | result included |
| `score1a`, `score1b`, `score2` | teacher-forced scoring: FRLP likelihood cells and residual-stream states | layer-24 states included |
| **`select`** (S6) | pick the TOP / ARC / SCATTER ban lists from the geodesic geometry of the algorithm centroids | **runs here** |
| **`landing`** (S10) | behaviour: landing shares, avoidance, gate, disguise, landing rank tests against the forecasts | **runs here** |
| **`manifold`** (S11) | activation geometry: PCA, k-NN geodesics, MDS / Isomap, manifold figures, persistent H1 | **runs here** |

This notebook re-runs the three CPU stages on a **curated 100-program subset** (`mini_demo_data.json`, loaded from GitHub):
- **56 forced programs**: 8 for each algorithm of the K = 7 analysis catalogue (BFS, DFS, bidirectional BFS, IDDFS, A*, greedy best-first, IDA*). They anchor the "algorithm manifold".
- **44 natural programs**: what the model chose to write under the baseline prompt, the single BFS ban and the four multi-name bans. The bans are TOP (BFS, DFS, greedy, IDDFS), ARC (BFS, BiBFS, A*, greedy), SCATTER (BFS, DFS, greedy, IDA*) and TOP5 (TOP + A*).
- For every program, its real **Qwen3-8B residual-stream state** at layer 24, positions `g16` and `pre` (4096-d, float16), taken from the teacher-forced pass. The data also carry the forecasts committed before sampling and the full-run reference numbers.

The code cells are the artifact's own code (`common.py`, `mb/sets.py`, `analysis/landing.py`, `analysis/manifold.py`, `method.py`) split into cells. File reads were replaced by reads from the loaded `data`, and hard-coded repetition counts moved to the config cell.

**Full-run headline (N = 100 programs per condition).** A single ban works: ban BFS and the model writes DFS 86 % of the time (avoidance 0.86). Every multi-ban fails. Avoidance is 0.13 for TOP, 0.37 for ARC, 0.04 for SCATTER and 0.00 for TOP5: the model re-implements the banned BFS or DFS, often without naming it.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru, ripser — NOT on Colab, always install (ripser also pulls in persim)
_pip('loguru==0.7.3', 'ripser==0.6.12')

# numpy, scipy, scikit-learn, matplotlib, Cython — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0', 'Cython==3.0.12')

In [ ]:
from __future__ import annotations

# --- original imports of method.py, common.py, mb/sets.py, analysis/landing.py and analysis/manifold.py ---
import argparse
import collections
import hashlib
import itertools
import json
import math
import os
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
from loguru import logger

# --- notebook additions: decode the shipped states, show the saved figures, plot the summary ---
import base64
import warnings

import matplotlib.pyplot as plt
from IPython.display import Image, display

# ripser warns whenever a point cloud has fewer points than dimensions (e.g. the 80 % subsamples of the 56 demo
# programs in PCA-50 space); the warning is only about input orientation, which is correct here
warnings.filterwarnings("ignore", message="The input point cloud has more columns than rows")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-295f8e-prompt-bans-substitute-not-just-subtract/fork/run_5eGZGmskluur/round-2/experiment-9/demo/mini_demo_data.json"
import http.client
import json
import urllib.request
from pathlib import Path

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=15) as response:
            remote = json.loads(response.read().decode())
        # Until this demo is pushed, the URL can still serve an older demo file with another layout.
        if isinstance(remote, dict) and "examples" in remote and "state_info" in remote:
            return remote
        print("GitHub data has an older layout, using local mini_demo_data.json")
    # Network errors, timeouts, a placeholder URL, bad JSON, a cut-off response.
    except (OSError, ValueError, http.client.HTTPException) as e:
        print("GitHub data unavailable, using local mini_demo_data.json:", e)
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(data["title"])
print(f"{len(data['examples'])} programs of dataset {data['dataset']}; model {data['model']} ({data['decoding_tag']})")
print("states:", {k: data["state_info"][k] for k in ("context", "layers", "positions", "dim", "dtype")})
for (kind, cond), n in sorted(collections.Counter((e["metadata_kind"], e["metadata_condition"]) for e in data["examples"]).items()):
    print(f"  {kind:<8} {cond:<20} {n}")

## A look at the data

Each example is one program written by Qwen3-8B, in the artifact's `method_out.json` format: `input` is the prompt, `output` is the mechanism label, and the `metadata_*` fields hold the condition, template, labeller votes, hidden-test result and code. Two kinds of program matter here:

- **forced** (`forced:<alg>`): the prompt asks for a named algorithm, e.g. *"Please write breadth-first search (BFS) for this task."* Only programs labelled as the requested algorithm are kept. They show where each algorithm lives in activation space.
- **natural**: the prompt is the plain task (`baseline`), or the task plus a ban written in one of five templates W1-W5, e.g. *"Do not use breadth-first search (BFS), bidirectional BFS, A* search or greedy best-first search for this task."* These are what the model *chose* to write.

`metadata_state_b64` holds the program's residual state: the program was **name-scrubbed** (identifiers such as `bfs` replaced) and teacher-forced through Qwen3-8B under a name-free baseline prompt (context `base`). The state was read at layer 24 at two token positions: `g16`, the 16th generated token, and `pre`, the token just before the first mechanism-evidence token (e.g. right before `deque` or `heapq`). Every program, forced or natural, is therefore placed in the same coordinates.

The cell below prints one natural program from the **ARC** ban. Its prompt bans BFS, bidirectional BFS, A* and greedy search, and the model wrote DFS, the survivor that every ladder-type forecast predicted.

In [ ]:
# One natural program of the ARC multi-ban: the prompt bans BFS, bidirectional BFS, A* and greedy best-first search.
ex = next(e for e in data["examples"] if e["metadata_condition"] == "multi:ARC" and e["output"] == "dfs")
print("PROMPT:\n" + ex["input"][:420] + " ...\n")
print("LABEL:", ex["output"], "| runtime tracer:", ex["metadata_runtime_label"], "| AST rules:", ex["metadata_ast_label"],
      "| labeller source:", ex["metadata_labeller_source"], "| passed hidden tests:", ex["metadata_passed"])
print("BANNED:", ex["metadata_banned"], "| SURVIVORS:", ex["metadata_survivors"])
print("committed top-1 forecasts:", {k: v for k, v in ex.items() if k.startswith("predict_")})
print("\nRESPONSE (first 25 lines):\n" + "\n".join(ex["metadata_text"].splitlines()[:25]))
vec = np.frombuffer(base64.b64decode(ex["metadata_state_b64"]["pre"]), dtype="<f2")
print(f"\nresidual state at layer {data['state_info']['layers'][0]}, position 'pre': {vec.shape[0]}-d float16, "
      f"norm {np.linalg.norm(vec.astype(np.float32)):.1f}")

## Configuration

Every repetition count that was hard-coded in the original modules is set here, with its original value in the comment. The geometry settings are also here: the primary setting (layer 24, position `pre`, k = 10) and the positions of the sweep. The full study sweeps layers 20-28 at four positions; the demo data ship layer 24 at `g16` and `pre` only. `STAGES` lists the `method.py` stages that the notebook runs.

In [ ]:
# ------------------------------------------------------------------------------------------------ demo configuration
# Every repetition count below was hard-coded in the original modules; the original value is in the comment.
STAGES = ["select", "landing", "manifold"]   # the CPU stages of method.py run by this notebook (S6, S10, S11)

# analysis/landing.py
N_BOOT = 200                   # template-cluster bootstrap reps per condition (original 2000)
# mb/sets.py: stage_select
K_SWEEP = (5, 10, 15, 20)      # k of the k-NN graph in the stability sweep; must contain PRIMARY["k"] (original (5, 10, 15, 20))
N_SELECT_BOOT = 20             # program-bootstrap resamples of the set selection (original 200)
# analysis/manifold.py
PRIMARY = {"layer": 24, "pos": "pre", "k": 10}   # primary geometry setting (original)
SWEEP_POSITIONS = ("g16", "pre")  # positions of the geometry sweep (original ("g16", "pre", "g8", "g32") at layers 20-28;
                                  # the demo data hold layer 24, positions g16 and pre)
N_MANIFOLD_BOOT = 20           # program-bootstrap resamples at the primary setting (original 200)
MDS_N_INIT = 4                 # SMACOF restarts of the metric MDS (original 8)
N_PERSIST_BOOT_PER_ALG = 20    # bootstrap centroids per algorithm in the persistent-H1 cloud (original 40)
N_PERSIST_SHUFFLE = 20         # label-shuffled null draws (original 200)
N_PERSIST_GAUSS = 10           # covariance-matched Gaussian null draws (original 100)
N_PERSIST_SUB = 5              # 80 % subsamples for the persistence fraction (original 50)

## Frozen protocol (`common.py`)

`common.py` holds the protocol that every stage reads:
- **The catalogues.** T1 has 10 grid algorithms, of which 7 form the analysis catalogue; T2 has 5 weighted shortest-path algorithms. Each has a frozen display name, the exact words used in the prompts, and a short `SHORT` tag for plots.
- **The alias lexicon.** It decides whether a program *names its own mechanism*. A banned-mechanism program that never uses any alias of its mechanism is **disguised**, e.g. DFS written as "backtracking".
- **The thresholds.** A multi-ban passes its **gate** when avoidance ≥ 0.5 *and* at least 20 attempted programs avoid the banned list. `other_floor` is the probability that every mixed forecast keeps for "other".
- **Helpers.** Logging, JSON I/O, the Wilson 95 % interval and the deviation log.

Only the parts the CPU stages use are copied. The task texts, prompt templates and hash-chain helpers stay in the repository's `common.py`. Results, figures and logs are written below the notebook's working directory, as below the repository root in the original.

In [ ]:
# common.py (excerpt): frozen protocol constants and shared helpers for the five-name ban study.
# The notebook's working directory plays the role of the repository root (ROOT = Path(__file__).resolve().parent).
ROOT = Path(".").resolve()
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "figures"
LOG_DIR = ROOT / "logs"

for _d in (RESULTS_DIR, FIG_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

MODEL_ID = os.environ.get("MB_MODEL", "Qwen/Qwen3-8B")
MODEL_REVISION = "b968826d9c46dd6066d109eabc6255188de91218"

TASKS = ("t1", "t2")
TASK_DATASET_NAME = {"t1": "T1_grid_path", "t2": "T2_weighted_shortest_path"}

# Catalogues (key -> frozen display name), in CANONICAL LIST ORDER.
CATALOGUE: dict[str, dict[str, str]] = {
    "t1": {
        "bfs": "breadth-first search (BFS)",
        "dfs": "depth-first search (DFS)",
        "bibfs": "bidirectional BFS",
        "iddfs": "iterative deepening DFS (IDDFS)",
        "astar": "A* search",
        "gbfs": "greedy best-first search",
        "dijkstra_unit": "unit-weight Dijkstra",
        "idastar": "IDA* search",
        "jps": "jump point search",
        "biastar": "bidirectional A* search",
    },
    "t2": {
        "dijkstra": "Dijkstra's algorithm",
        "bf_spfa": "Bellman-Ford or SPFA",
        "bidijkstra": "bidirectional Dijkstra",
        "floyd": "Floyd-Warshall",
        "dial": "Dial's bucket-queue algorithm",
    },
}
MODES = {t: list(CATALOGUE[t].keys()) for t in TASKS}
NEW_T1 = ("gbfs", "idastar", "jps", "biastar")
OTHER = "other"
UNKNOWN = "unknown"
NO_CODE = "no_code"
SHORT = {"bfs": "BFS", "dfs": "DFS", "bibfs": "BiBFS", "iddfs": "IDDFS", "astar": "A*", "gbfs": "GBFS",
         "dijkstra_unit": "Dijkstra(u)", "idastar": "IDA*", "jps": "JPS", "biastar": "BiA*",
         "dijkstra": "Dijkstra", "bf_spfa": "BF/SPFA", "bidijkstra": "BiDijkstra", "floyd": "Floyd",
         "dial": "Dial", "other": "other", "unknown": "unknown", "no_code": "no code"}

# Alias lexicon (case-insensitive substrings) for disguise and scrub steps; keyed by mechanism family.
ALIASES: dict[str, list[str]] = {
    "bfs": ["bfs", "breadth-first", "breadth first", "breadth_first"],
    "dfs": ["dfs", "depth-first", "depth first", "depth_first"],
    "astar": ["a*", "a-star", "astar", "a_star"],
    "gbfs": ["greedy", "best-first", "best first", "best_first"],
    "iddfs": ["iddfs", "iterative deepening", "iterative_deepening"],
    "idastar": ["ida*", "ida_star", "idastar"],
    "jps": ["jps", "jump point", "jump_point"],
    "bidir": ["bidirectional", "bidir"],
    "dijkstra": ["dijkstra"],
    "bf_spfa": ["bellman", "ford", "spfa"],
    "floyd": ["floyd", "warshall"],
    "dial": ["dial", "bucket queue", "bucket_queue"],
}
# Which alias families name each mechanism (a program 'names its own mechanism' if any family matches).
MECH_ALIAS_FAMILIES: dict[str, list[str]] = {
    "bfs": ["bfs"], "dfs": ["dfs"], "bibfs": ["bidir"], "iddfs": ["iddfs"], "astar": ["astar"], "gbfs": ["gbfs"],
    "dijkstra_unit": ["dijkstra"], "idastar": ["idastar"], "jps": ["jps"], "biastar": ["bidir"],
    "dijkstra": ["dijkstra"], "bf_spfa": ["bf_spfa"], "bidijkstra": ["bidir"], "floyd": ["floyd"], "dial": ["dial"],
}

THRESHOLDS = {"keep_forced_share": 0.70, "smoke_descriptive_min": 7, "gate_avoidance": 0.50, "gate_min_nonbanned": 20,
              "uniform_mix": 0.05, "other_floor": 0.02, "ess_flag": 5.0, "gold_accept": 36, "gold_per_algo_min": 3}
DECODING = {"temperature": 1.0, "top_p": 1.0, "top_k": -1, "max_tokens": 1024, "thinking": "off"}
DECODING_TAG = "T1.0_topp1.0_topk-off_max1024_thinkoff"
LAYERS = list(range(20, 29))
GEN_POSITIONS = ("g8", "g16", "g32", "pre")
N_PER_TEMPLATE = 20
M_MAX = 30


def setup_logging(name: str) -> None:
    """loguru: stdout INFO + rotating DEBUG file sink per stage."""
    logger.remove()
    logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
    logger.add(str(LOG_DIR / f"{name}.log"), rotation="30 MB", level="DEBUG")


def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


def write_json(p: Path, obj: Any, indent: int | None = 1, sort_keys: bool = False) -> None:
    p = Path(p)
    tmp = p.with_suffix(p.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=indent, sort_keys=sort_keys, ensure_ascii=False, default=_json_default))
    tmp.replace(p)


def read_json(p: Path) -> Any:
    return json.loads(Path(p).read_text())


def _json_default(o: Any) -> Any:
    try:
        import numpy as np  # local import: keeps the sandbox runner free of numpy

        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return None if not np.isfinite(o) else float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, np.bool_):
            return bool(o)
    except ImportError:
        pass
    if isinstance(o, (set, frozenset)):
        return sorted(o)
    if isinstance(o, Path):
        return str(o)
    raise TypeError(f"not JSON serialisable: {type(o)}")


def wilson_ci(k: int, n: int, z: float = 1.959963984540054) -> tuple[float, float]:
    if n <= 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, centre - half), min(1.0, centre + half))


def deviation(code: str, text: str) -> None:
    """Append one entry to results/deviations.json (deduplicated by code)."""
    p = RESULTS_DIR / "deviations.json"
    devs = json.loads(p.read_text()) if p.exists() else []
    if any(d["code"] == code for d in devs):
        return
    devs.append({"code": code, "text": text, "utc": utc_now()})
    write_json(p, devs)
    logger.warning(f"DEVIATION {code}: {text}")

## Feeding the stages from `data`

The original stages read their inputs from the repository's files. Here two small adapters build the same objects from `data`, so that the stage code below can run unchanged:

| original input | in this notebook |
|---|---|
| `data/labelled_session1.jsonl`, `data/labelled_session2.jsonl` (one labelled row per natural program) | `NATURAL_ROWS`: the natural examples with the `metadata_` prefix dropped and `label = output` |
| `states/<task>/<context>/part_*.npz`, read by `mb/score.py: load_states()` | `load_states()` below: the same dict `{uids, X[program, layer, position, 4096], meta, R}`, decoded from the base64 float16 vectors |
| `predictions/forecasts_<SET>.json`, `results/catalogue_keep.json` | `data["forecasts"][SET]`, `data["catalogue_keep"]` |

Only context `base` of T1 is shipped. Positions not shipped (`g8`, `g32`) stay NaN, and the original `ok = ~np.isnan(...)` masks would skip them. Asking for another context (e.g. `forced`) raises `ValueError`, which `stage_select` already catches and records as an error. `LAYERS` is narrowed to the shipped layer 24.

In [ ]:
# Inputs of the CPU stages, rebuilt from `data` (see the table above).
LAYERS = list(data["state_info"]["layers"])  # original list(range(20, 29)); the demo data hold layer 24 only
POS_NAMES = GEN_POSITIONS                    # mb/score.py: position order of the state arrays

# data/labelled_session{1,2}.jsonl -> one labelled row per natural program
NATURAL_ROWS = [{**{k.removeprefix("metadata_"): v for k, v in e.items() if k != "metadata_state_b64"},
                 "label": e["output"]} for e in data["examples"] if e["metadata_kind"] == "natural"]


def load_states(task: str, context: str) -> dict[str, Any]:
    """mb/score.py load_states() for the demo data: X[program, layer, position, 4096] (float16) and one meta dict per
    program, as in states/<task>/<context>/part_*.npz. Positions that are not shipped stay NaN."""
    info = data["state_info"]
    if task != "t1" or context != info["context"]:
        raise ValueError(f"states {task}/{context} are not in the demo data (only t1/{info['context']})")
    exs = data["examples"]
    X = np.full((len(exs), len(LAYERS), len(POS_NAMES), info["dim"]), np.nan, dtype=np.float16)
    li = LAYERS.index(info["layers"][0])
    uids, metas = [], []
    for i, e in enumerate(exs):
        for pos, b64 in e["metadata_state_b64"].items():
            X[i, li, POS_NAMES.index(pos), :] = np.frombuffer(base64.b64decode(b64), dtype="<f2")
        m = {"uid": e["metadata_state_uid"], "task": e["metadata_task"], "label": e["output"], "context": context,
             "kind": e["metadata_kind"], "pre_fallback": e["metadata_pre_fallback"]}
        if e["metadata_kind"] == "forced":
            m["algorithm"] = e["metadata_requested_algorithm"]
        else:
            m.update(condition=e["metadata_condition"], template=e["metadata_template"])
        uids.append(m["uid"])
        metas.append(m)
    return {"uids": uids, "X": X, "meta": metas, "R": {}}


_st = load_states("t1", "base")
print("X:", _st["X"].shape, _st["X"].dtype, "| forced:", sum(m["kind"] == "forced" for m in _st["meta"]),
      "| natural:", sum(m["kind"] == "natural" for m in _st["meta"]), "| natural rows:", len(NATURAL_ROWS))

## S6 · Set selection (`mb/sets.py`), part 1: geodesic distances between algorithm centroids

How far apart are two algorithms *inside the model*? The study uses the model's own forced programs as points: each forced BFS program is one 4096-d residual state, each forced DFS program another, and so on. The steps for one (layer, position) setting:

1. `pca` centres the states and keeps the first 50 principal components.
2. `knn_geodesic` builds a symmetric k-NN graph (k = 10) with Euclidean edge weights. Disconnected components are bridged by their shortest edge, and every bridge is logged. Shortest paths in this graph approximate **geodesic** distances along the data manifold rather than straight-line distances through empty space.
3. `centroid_matrix` sets `D[a, b]` to the median geodesic distance between the points of algorithm *a* and the points of algorithm *b*, divided by the median off-diagonal entry so that settings are comparable.

In [ ]:
# mb/sets.py -- S6 SET SELECTION: TOP-5 from FRLP, ARC-5 / SCATTER-5 from the consensus geodesic matrix of algorithm
# centroids.
#
# Geometry per (layer, position) in context 'base' (positions g16 and pre, layers 20-28): centre, PCA-50 on all T1
# forced points, symmetric k-NN graph (k=10, Euclidean weights; disconnected components bridged by their shortest
# Euclidean edge, logged), Dijkstra geodesics, D[a,b] = median geodesic over point pairs of a and b, normalised by its
# median off-diagonal. Consensus Dbar = elementwise median over the 18 settings. Seriation = exact Held-Karp shortest
# Hamiltonian path on Dbar.
POS_INDEX = {"g8": 0, "g16": 1, "g32": 2, "pre": 3}
PRIMARY_POS = ("g16", "pre")
# K_SWEEP: see the config cell (original (5, 10, 15, 20))


def pca(X: np.ndarray, n: int = 50) -> np.ndarray:
    Xc = X - X.mean(axis=0, keepdims=True)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    n = min(n, Vt.shape[0])
    return Xc @ Vt[:n].T


def knn_geodesic(Z: np.ndarray, k: int = 10) -> tuple[np.ndarray, dict[str, Any]]:
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import connected_components, shortest_path
    from scipy.spatial.distance import cdist

    n = Z.shape[0]
    E = cdist(Z, Z)
    k = min(k, n - 1)
    idx = np.argsort(E, axis=1)[:, 1:k + 1]
    W = np.zeros((n, n))
    rows = np.repeat(np.arange(n), k)
    W[rows, idx.ravel()] = E[rows, idx.ravel()]
    W = np.maximum(W, W.T)
    info = {"k": k, "n": n, "bridges": 0}
    ncomp, lab = connected_components(csr_matrix(W > 0), directed=False)
    while ncomp > 1:
        best = None
        for c in range(1, ncomp):
            a, b = np.where(lab == 0)[0], np.where(lab == c)[0]
            sub = E[np.ix_(a, b)]
            i, j = np.unravel_index(np.argmin(sub), sub.shape)
            if best is None or sub[i, j] < best[0]:
                best = (sub[i, j], a[i], b[j])
        W[best[1], best[2]] = W[best[2], best[1]] = best[0]
        info["bridges"] += 1
        ncomp, lab = connected_components(csr_matrix(W > 0), directed=False)
    info["connected_without_bridges"] = info["bridges"] == 0
    G = shortest_path(csr_matrix(W), method="D", directed=False)
    return G, info


def centroid_matrix(G: np.ndarray, labels: np.ndarray, algs: list[str],
                    sel: dict[str, np.ndarray] | None = None) -> np.ndarray:
    K = len(algs)
    D = np.zeros((K, K))
    ids = {a: (sel[a] if sel is not None else np.where(labels == a)[0]) for a in algs}
    for i, a in enumerate(algs):
        for j, b in enumerate(algs):
            sub = G[np.ix_(ids[a], ids[b])]
            if i == j:
                m = sub[~np.eye(len(ids[a]), dtype=bool)] if len(ids[a]) > 1 else sub.ravel()
                D[i, j] = np.median(m) if m.size else 0.0
            else:
                D[i, j] = np.median(sub)
    off = D[~np.eye(K, dtype=bool)]
    return D / np.median(off)

### Part 2: seriation and the rules that turn geometry into ban lists

- `held_karp_path` finds the exact shortest Hamiltonian path through the K = 7 centroids, a 1-D ordering of the algorithms along the manifold (seriation). `brute_force_path` checks it.
- **ARC** is the window of `size` consecutive algorithms along that path with the smallest summed pairwise distance, excluding the TOP set: a tight, contiguous arc of neighbours.
- **SCATTER** is the set of `size` algorithms whose *closest* pair is as far apart as possible: a ban spread over the whole manifold.
- `edge_opposite`: for a banned set, the survivors are ordered by their distance to the nearest banned centroid. **EDGE** is the closest one or two, **OPPOSITE** the farthest. The edge hypothesis predicts that the model lands next to what it was told not to write.

In [ ]:
def held_karp_path(D: np.ndarray) -> tuple[list[int], float]:
    """Exact shortest Hamiltonian path (free endpoints) by Held-Karp DP."""
    K = D.shape[0]
    INF = float("inf")
    dp = np.full((1 << K, K), INF)
    par = np.full((1 << K, K), -1, dtype=int)
    for i in range(K):
        dp[1 << i, i] = 0.0
    for mask in range(1 << K):
        for last in range(K):
            v = dp[mask, last]
            if v == INF or not (mask >> last) & 1:
                continue
            for nxt in range(K):
                if (mask >> nxt) & 1:
                    continue
                m2 = mask | (1 << nxt)
                c = v + D[last, nxt]
                if c < dp[m2, nxt]:
                    dp[m2, nxt] = c
                    par[m2, nxt] = last
    full = (1 << K) - 1
    last = int(np.argmin(dp[full]))
    cost = float(dp[full, last])
    path, mask = [], full
    while last != -1:
        path.append(last)
        p = par[mask, last]
        mask ^= 1 << last
        last = int(p)
    return path[::-1], cost


def brute_force_path(D: np.ndarray) -> float:
    K = D.shape[0]
    return min(sum(D[p[i], p[i + 1]] for i in range(K - 1)) for p in itertools.permutations(range(K)))


def pair_sum(D: np.ndarray, S: list[int]) -> float:
    return float(sum(D[a, b] for a, b in itertools.combinations(S, 2)))


def pair_min(D: np.ndarray, S: list[int]) -> float:
    return float(min(D[a, b] for a, b in itertools.combinations(S, 2)))


def select_sets(D: np.ndarray, algs: list[str], top: list[str], size: int) -> dict[str, Any]:
    path, cost = held_karp_path(D)
    K = len(algs)
    top_idx = sorted(algs.index(a) for a in top)
    wins = []
    for s in range(0, K - size + 1):
        w = sorted(path[s:s + size])
        if w == top_idx:
            continue
        wins.append((pair_sum(D, w), s, w))
    arc = min(wins)[2] if wins else None
    scat = None
    best = None
    for S in itertools.combinations(range(K), size):
        S = sorted(S)
        if S == top_idx or (arc is not None and S == arc):
            continue
        key = (pair_min(D, S), pair_sum(D, S))
        if best is None or key > best:
            best, scat = key, S
    return {"path": [algs[i] for i in path], "path_cost": cost,
            "ARC": [algs[i] for i in arc] if arc is not None else None,
            "SCATTER": [algs[i] for i in scat] if scat is not None else None}


def edge_opposite(D: np.ndarray, algs: list[str], banned: list[str]) -> dict[str, Any]:
    bi = [algs.index(b) for b in banned]
    surv = [a for a in algs if a not in banned]
    dmin = {a: float(min(D[algs.index(a), j] for j in bi)) for a in surv}
    dmean = {a: float(np.mean([D[algs.index(a), j] for j in bi])) for a in surv}
    order = sorted(surv, key=lambda a: (dmin[a], dmean[a], a))
    n2 = 1 if len(surv) == 2 else min(2, len(surv))
    return {"survivors": surv, "d_min": dmin, "d_mean": dmean, "EDGE": order[:n2], "OPPOSITE": order[::-1][:n2],
            "order_near_to_far": order}

### Part 3: geometry over settings and the FRLP ladder

- `geometry` repeats part 1 for every (layer, position) setting and takes the element-wise median, `Dbar`. The study used 9 layers × 2 positions = 18 settings and 182 forced programs. The demo has layer 24 × 2 positions = 2 settings and 56 programs.
- `frlp_base` returns the **forced-ratio latent probability** of each algorithm under the plain prompt. FRLP is an importance-sampling estimate of the probability that the model writes an algorithm-*j* program unprompted, `L_j = log mean_programs exp(log p(program | plain prompt) − log p(program | forced prompt))`, computed by teacher forcing each forced *j*-program under both prompts. The FRLP cells (≈ 17k forward passes) are GPU output; the demo data carry their base values.
- `corrected_L` adds `log(share of forced programs labelled as requested)`, so that an algorithm the model cannot write reliably is not overrated. Sorting by corrected L gives the **ladder**: BFS ≈ 0.6, DFS ≈ −0.5, greedy ≈ −18.6, IDDFS ≈ −30.9, A* ≈ −32.1, BiBFS ≈ −47.6, IDA* ≈ −129.1 (nats). The TOP sets are its first four or five rungs.

In [ ]:
def load_points(task: str, context: str, algs: list[str]) -> tuple[np.ndarray, np.ndarray, list[dict[str, Any]]]:
    # from mb.score import load_states   (defined above, from the demo data)

    st = load_states(task, context)
    keep = [i for i, m in enumerate(st["meta"]) if m.get("kind") == "forced" and m.get("algorithm") in algs]
    X = st["X"][keep].astype(np.float32)
    labels = np.array([st["meta"][i]["algorithm"] for i in keep])
    return X, labels, [st["meta"][i] for i in keep]


def geometry(task: str, context: str, algs: list[str], k: int = 10, positions=PRIMARY_POS,
             layers=None) -> dict[str, Any]:
    X, labels, meta = load_points(task, context, algs)
    layers = layers or LAYERS
    mats, infos, settings, Gs = [], [], [], {}
    for li, l in enumerate(LAYERS):
        if l not in layers:
            continue
        for pos in positions:
            Y = X[:, li, POS_INDEX[pos], :]
            ok = ~np.isnan(Y).any(axis=1)
            Z = pca(Y[ok])
            G, info = knn_geodesic(Z, k)
            Dm = centroid_matrix(G, labels[ok], algs)
            mats.append(Dm)
            infos.append({"layer": l, "position": pos, **info, "n_points": int(ok.sum())})
            settings.append((l, pos))
            Gs[(l, pos)] = (G, labels[ok])
    Dbar = np.median(np.stack(mats), axis=0)
    return {"Dbar": Dbar, "mats": mats, "settings": settings, "infos": infos, "Gs": Gs, "labels": labels}


def frlp_base(task: str) -> dict[str, dict[str, float]]:
    # original: rows = [r for r in read_jsonl(RESULTS_DIR / "frlp_cells.jsonl") if r["task"] == task and
    #                   r["condition"] == "base" and r["half"] == "all" and "variant" not in r]
    # results/frlp_cells.jsonl is GPU output; the demo data carry the L values of these base-condition cells.
    rows = [{"algorithm": a, "L": L} for a, L in data["committed_geometry"]["L_base_raw"].items()] if task == "t1" else []
    return {r["algorithm"]: r for r in rows}


def corrected_L(task: str, L: dict[str, float]) -> dict[str, float]:
    """L_j + log(as-requested share of F_j): removes the P(j | F_j) < 1 bias in cross-algorithm comparisons."""
    keep = data["catalogue_keep"]["tasks"][task]  # read_json(RESULTS_DIR / "catalogue_keep.json")["tasks"][task]
    return {a: L[a] + math.log(max(keep[a]["share"], 1e-9)) for a in L}

### Part 4: `stage_select`, the S6 stage itself

`stage_select` does the following:
- takes the K = 7 analysis catalogue;
- ranks it by corrected FRLP (TOP = first 4, because K = 7 gives 4-name sets);
- computes `Dbar` and derives ARC and SCATTER;
- adds the literal five-name ban TOP5, logged as deviation `D-top5-literal`;
- records EDGE / OPPOSITE for every set;
- measures stability over every (k, setting) and over `N_SELECT_BOOT` program-bootstrap resamples;
- checks Held-Karp against brute force.

Two parts cannot run here and are left as in the original: the context-sensitivity check, which needs the `forced`-context states and is caught and recorded as an error, and the planned-conditions file, which needs the frozen prompts and only feeds the GPU stages.

The sets that were actually sampled are the **committed** ones, computed on all 182 forced programs and 18 settings. The demo recomputes ARC and SCATTER on its 56-program subset and compares them with the committed sets after the stage runs.

In [ ]:
def stage_select() -> None:
    # from mb.prompts import load_frozen, multi_ban_conditions   (only for the planned-conditions file at the end)

    keep = data["catalogue_keep"]  # read_json(RESULTS_DIR / "catalogue_keep.json")
    algs = [a for a in keep["analysis_catalogue_final"]["t1"]]
    fb = frlp_base("t1")
    algs = [a for a in algs if a in fb]
    K = len(algs)
    L_raw = {a: fb[a]["L"] for a in algs}
    L_cor = corrected_L("t1", L_raw)
    size = 5 if K >= 8 else (4 if K == 7 else 5)
    top = sorted(algs, key=lambda a: -L_cor[a])[:size]
    geo = geometry("t1", "base", algs)
    Dbar = geo["Dbar"]
    sel = select_sets(Dbar, algs, top, size) if K > size else {"path": algs, "ARC": None, "SCATTER": None}
    sets = {"TOP": top}
    if sel.get("ARC"):
        sets["ARC"] = sel["ARC"]
    if sel.get("SCATTER"):
        sets["SCATTER"] = sel["SCATTER"]
    if size == 4 and K >= 7:
        # the literal request: the model's top FIVE in one prompt (added beside F4's TOP-4 / ARC-4 / SCATTER-4)
        sets["TOP5"] = sorted(algs, key=lambda a: -L_cor[a])[:5]
        # from common import deviation   (defined above)

        deviation("D-top5-literal", "With K = 7 the plan's F4 runs TOP-4, ARC-4 and SCATTER-4 (3 survivors each); the "
                                    "literal five-name ban TOP5 (top five by corrected FRLP, 2 survivors) is added as a "
                                    "fourth sampled condition with the same templates, twins and committed forecasts.")
    med = float(np.median(Dbar[~np.eye(K, dtype=bool)]))
    eo = {s: edge_opposite(Dbar, algs, b) for s, b in sets.items()}
    # stability: every single (layer, position, k) setting and N_SELECT_BOOT (original 200) program-bootstrap resamples
    stab = {"per_setting": [], "bootstrap": {}}
    X, labels, _ = load_points("t1", "base", algs)
    for k in K_SWEEP:
        gk = geometry("t1", "base", algs, k=k) if k != 10 else geo
        for (l, pos), Dm in zip(gk["settings"], gk["mats"]):
            s2 = select_sets(Dm, algs, top, size) if K > size else {}
            stab["per_setting"].append({"k": k, "layer": l, "position": pos, "ARC": s2.get("ARC"),
                                        "SCATTER": s2.get("SCATTER"),
                                        "EDGE": {s: edge_opposite(Dm, algs, b)["EDGE"] for s, b in sets.items()},
                                        "OPPOSITE": {s: edge_opposite(Dm, algs, b)["OPPOSITE"] for s, b in sets.items()}})
    rng = np.random.default_rng(0)
    agree = {s: {"EDGE": 0, "OPPOSITE": 0} for s in sets}
    arc_same = scat_same = 0
    for _ in range(N_SELECT_BOOT):
        mats = []
        for (l, pos), (G, lab) in geo["Gs"].items():
            selidx = {a: rng.choice(np.where(lab == a)[0], size=(lab == a).sum(), replace=True) for a in algs}
            mats.append(centroid_matrix(G, lab, algs, selidx))
        Db = np.median(np.stack(mats), axis=0)
        if K > size:
            s2 = select_sets(Db, algs, top, size)
            arc_same += int(s2["ARC"] is not None and sets.get("ARC") is not None and sorted(s2["ARC"]) == sorted(sets["ARC"]))
            scat_same += int(s2["SCATTER"] is not None and sets.get("SCATTER") is not None
                             and sorted(s2["SCATTER"]) == sorted(sets["SCATTER"]))
        for s, b in sets.items():
            e2 = edge_opposite(Db, algs, b)
            agree[s]["EDGE"] += int(e2["EDGE"] == eo[s]["EDGE"])
            agree[s]["OPPOSITE"] += int(e2["OPPOSITE"] == eo[s]["OPPOSITE"])
    stab["bootstrap"] = {"n": N_SELECT_BOOT, "ARC_same": arc_same / N_SELECT_BOOT, "SCATTER_same": scat_same / N_SELECT_BOOT,
                         "EDGE_OPPOSITE_same": {s: {k2: v / N_SELECT_BOOT for k2, v in d.items()} for s, d in agree.items()}}
    # context sensitivity: 'forced' context and R of the forced prompts
    ctx = {}
    try:
        gf = geometry("t1", "forced", algs)
        sf = select_sets(gf["Dbar"], algs, top, size) if K > size else {}
        ctx["forced"] = {"Dbar": gf["Dbar"], "ARC": sf.get("ARC"), "SCATTER": sf.get("SCATTER"),
                         "EDGE": {s: edge_opposite(gf["Dbar"], algs, b)["EDGE"] for s, b in sets.items()}}
    except (ValueError, IndexError) as e:
        ctx["forced"] = {"error": str(e)}
    try:
        ctx["R_forced_prompts"] = r_geometry(algs, top, size, sets)
    except (ValueError, IndexError, KeyError) as e:
        ctx["R_forced_prompts"] = {"error": str(e)}

    def jac(a, b):
        if not a or not b:
            return None
        return len(set(a) & set(b)) / len(set(a) | set(b))

    for c in ctx.values():
        if "error" in c:
            continue
        c["jaccard"] = {"ARC": jac(c.get("ARC"), sets.get("ARC")), "SCATTER": jac(c.get("SCATTER"), sets.get("SCATTER"))}
        c["EDGE_same"] = {s: c["EDGE"].get(s) == eo[s]["EDGE"] for s in sets}
    # seriation check vs brute force on K = 7 (or K)
    kk = min(7, K)
    hk = held_karp_path(Dbar[:kk, :kk])[1]
    bf = brute_force_path(Dbar[:kk, :kk])
    out = {"task": "t1", "algorithms": algs, "K": K, "set_size": size, "L_base_raw": L_raw, "L_base_corrected": L_cor,
           "sets": sets, "seriation_path": sel.get("path"), "Dbar": Dbar, "median_offdiag": med,
           "edge_opposite": eo, "settings": [{"layer": l, "position": p} for l, p in geo["settings"]],
           "graph_info": geo["infos"], "stability": stab, "context_sensitivity": ctx,
           "held_karp_vs_bruteforce": {"K": kk, "held_karp": hk, "brute_force": bf, "match": abs(hk - bf) < 1e-9}}
    write_json(RESULTS_DIR / "sets_and_geometry.json", out)
    # planned conditions (frozen rule) for score1b and session 2: they need the frozen prompts and only feed the GPU
    # stages, so they are not rebuilt here (the sampled sets are the committed ones in data["forecasts"])
    # fz = load_frozen()
    # plan = {"t1": {}}
    # for s, b in sets.items():
    #     plan["t1"][s] = {"banned": b, "survivors": [a for a in algs if a not in b],
    #                      "templates": multi_ban_conditions("t1", s, b, fz["neutral_nouns"]["t1"])}
    # write_json(RESULTS_DIR / "planned_conditions.json", plan)
    logger.info(f"K={K} algorithms {algs}; L_cor {json.dumps({a: round(v, 2) for a, v in L_cor.items()})}")
    logger.info(f"sets: {sets}; path {sel.get('path')}")
    for s in sets:
        logger.info(f"  {s}: survivors {eo[s]['survivors']} EDGE {eo[s]['EDGE']} OPPOSITE {eo[s]['OPPOSITE']}")


def r_geometry(algs: list[str], top: list[str], size: int, sets: dict[str, list[str]]) -> dict[str, Any]:
    # from mb.score import load_states   (defined above, from the demo data)

    st = load_states("t1", "forced")
    keys = [k for k in st["R"] if k.startswith("t1|forced:")]
    lab = np.array([k.split("|")[1].split(":")[1] for k in keys])
    keep = [i for i, a in enumerate(lab) if a in algs]
    R = np.stack([st["R"][keys[i]] for i in keep]).astype(np.float32)
    lab = lab[keep]
    mats = []
    for li, _l in enumerate(LAYERS):
        Z = pca(R[:, li, :], n=min(50, len(lab) - 1))
        G, _ = knn_geodesic(Z, k=5)
        mats.append(centroid_matrix(G, lab, algs))
    D = np.median(np.stack(mats), axis=0)
    s = select_sets(D, algs, top, size) if len(algs) > size else {}
    return {"Dbar": D, "ARC": s.get("ARC"), "SCATTER": s.get("SCATTER"),
            "EDGE": {k: edge_opposite(D, algs, b)["EDGE"] for k, b in sets.items()}, "k": 5,
            "note": "R = last prompt token of the 10 forced prompts per algorithm (carries the name)"}

## S10 · Behaviour and landing tests (`analysis/landing.py`), part 1: statistics

This module was frozen and **sha256-committed before any multi-ban sample was drawn**, together with six forecasts per banned set of where the model would land: LADDER, PRIME_AND_VETO, PROXIMITY, FRLP_BAN, GENERIC and CORPUS. The statistics helpers:
- `spearman`: rank correlation between a forecast and the realised survivor shares.
- `perm_null`: an **exact** permutation null that re-assigns the realised survivor shares over all S! orderings. With three survivors there are only 6 orderings, so the smallest attainable p is 1/6.
- `product_null`: pools several banned sets with the exact product of their permutation nulls.
- `logscore`: mean log-probability that a mixed forecast assigns to the programs actually written.

In [ ]:
# analysis/landing.py -- S10 BEHAVIOUR (frozen and sha256-committed at S7, before any multi-ban sample).
#
# Per sampled condition: attempted programs (extracted code with a known label); landing distribution over {each survivor,
# banned (by algorithm), other}; avoidance = 1 - banned share; GATE = avoidance >= 0.5 and >= 20 attempted non-banned
# programs; compliant share (passed and not banned); unknown and truncation rates. Wilson 95% CIs, per-template tables
# and template-cluster bootstrap (2,000 reps). DISGUISE: among banned-mechanism programs, the share whose full response
# matches no alias of its own mechanism, with the self-names used. LANDING TEST per banned set on the landed survivor
# mass: S_edge = share(EDGE) - share(OPPOSITE); Spearman rho of survivor shares with each committed forecast; exact
# survivor-permutation nulls (all S! assignments, one-sided in the predicted direction), pooled over sets with the exact
# product null; log score of each committed mixed forecast against uniform, and top-1 hits; the same rank tests on the
# likelihood landing (mean_w G_h2) against PV from half 1. References: single modal bans. T2 leave-one-survivor: will
# versus can, FRLP gain and banned drop.
FORECASTS = ("LADDER", "PRIME_AND_VETO", "PROXIMITY", "FRLP_BAN", "GENERIC", "CORPUS")
# N_BOOT: see the config cell (original 2000)
UNRESOLVED = ("unknown", "no_code", "inconsistent")
SELF_NAME_RE = re.compile(r"\b(backtrack\w*|flood[\s_-]?fill\w*|wave[\s_-]?front|level[\s_-]?order|breadth[\s_-]?first|"
                          r"depth[\s_-]?first|best[\s_-]?first|greedy|heuristic|a\*|a[\s_-]?star|dijkstra|bfs|dfs|"
                          r"iddfs|ida\*|bidirectional|bellman[\s_-]?ford|spfa|floyd|dial|bucket|priority[\s_-]?queue|"
                          r"recursive|stack[\s_-]?based|queue[\s_-]?based|random[\s_-]?walk|wall[\s_-]?follow\w*|"
                          r"spiral|brute[\s_-]?force|exhaustive|depth[\s_-]?limited|iterative[\s_-]?deepening|"
                          r"label[\s_-]?correcting|relaxation|memoi[sz]\w*|dynamic[\s_-]?programming)\b", re.I)


# ------------------------------------------------------------------------------------------------ statistics
def spearman(x: list[float], y: list[float]) -> float | None:
    from scipy.stats import spearmanr

    if len(x) < 2 or len(set(x)) < 2 or len(set(y)) < 2:
        return None
    r = spearmanr(x, y).statistic
    return None if r is None or not np.isfinite(r) else float(r)


def perm_null(values: list[float], stat, observed: float) -> dict[str, Any]:
    """Exact survivor-permutation null: permute the realised survivor values over all S! assignments."""
    perms = list(itertools.permutations(values))
    dist = [stat(list(p)) for p in perms]
    dist = [d for d in dist if d is not None]
    if observed is None or not dist:
        return {"p": None, "n_perm": len(perms)}
    p = sum(1 for d in dist if d >= observed - 1e-12) / len(dist)
    return {"p": p, "n_perm": len(perms), "null_mean": float(np.mean(dist))}


def product_null(per_set: list[tuple[list[float], Any, float]]) -> dict[str, Any]:
    """Pooled exact product null: sum of per-set statistics over the product of the per-set permutation sets."""
    dists = []
    obs = 0.0
    for vals, stat, o in per_set:
        d = [stat(list(p)) for p in itertools.permutations(vals)]
        d = [x for x in d if x is not None]
        if o is None or not d:
            continue
        dists.append(d)
        obs += o
    if not dists:
        return {"p": None}
    tot = np.zeros(1)
    for d in dists:
        tot = (tot[:, None] + np.array(d)[None, :]).ravel()
    return {"p": float((tot >= obs - 1e-12).mean()), "n": int(tot.size), "observed_sum": obs}


def logscore(probs: dict[str, float], labels: list[str]) -> float:
    return float(np.mean([math.log(max(probs.get(y, probs.get("other", 1e-9)), 1e-12)) for y in labels])) if labels else float("nan")

### Part 2: per-condition landing summary and disguise

For each sampled condition, `summarise` does the following:
- keeps the **attempted** programs (code extracted, label known);
- maps each one to a landing category: `banned:<alg>`, a surviving catalogue algorithm, or `other`;
- reports the counts and shares with Wilson 95 % intervals;
- computes **avoidance** = 1 − banned share, the **gate** (avoidance ≥ 0.5 and ≥ 20 non-banned programs), the compliant share (passed the hidden tests and not banned), per-template tables, and a **template-cluster bootstrap** that resamples whole templates (`N_BOOT` reps), because the ban wording is a large random effect.

`disguise` looks at the banned-mechanism programs only and counts those whose full response never names their own mechanism (`names_own_mechanism`, using the alias lexicon). It also lists the names those programs gave themselves (`self_names`): function names and words such as "backtracking" or "flood fill".

In [ ]:
# ------------------------------------------------------------------------------------------------ landing per condition
def landing_category(label: str, banned: list[str], catalogue: list[str]) -> str:
    if label in banned:
        return "banned:" + label
    if label in catalogue:
        return label
    return "other"


def summarise(rows: list[dict[str, Any]], banned: list[str], catalogue: list[str], survivors: list[str],
              rng: np.random.Generator) -> dict[str, Any]:
    n_all = len(rows)
    att = [r for r in rows if r.get("label") not in UNRESOLVED and r.get("code")]
    cats = [landing_category(r["label"], banned, catalogue) for r in att]
    cnt = collections.Counter(cats)
    n = len(att)
    nb = sum(v for k, v in cnt.items() if k.startswith("banned:"))
    nonb = n - nb
    avoid = 1 - nb / n if n else None
    comp = sum(1 for r, c in zip(att, cats) if r.get("passed") and not c.startswith("banned:"))
    out = {"n_samples": n_all, "attempted": n, "unknown_rate": (n_all - n) / max(n_all, 1),
           "truncation_rate": sum(1 for r in rows if r.get("truncated")) / max(n_all, 1),
           "counts": dict(cnt), "fine_labels": dict(collections.Counter(r["label"] for r in rows)),
           "shares": {k: v / n for k, v in cnt.items()} if n else {},
           "share_ci": {k: wilson_ci(v, n) for k, v in cnt.items()},
           "avoidance": avoid, "avoidance_ci": wilson_ci(nonb, n) if n else None,
           "n_nonbanned": nonb, "compliant_share": comp / max(n_all, 1), "compliant_ci": wilson_ci(comp, n_all),
           "gate": bool(avoid is not None and avoid >= THRESHOLDS["gate_avoidance"]
                        and nonb >= THRESHOLDS["gate_min_nonbanned"]),
           "pass_rate_attempted": sum(1 for r in att if r.get("passed")) / max(n, 1)}
    surv_counts = {s: cnt.get(s, 0) for s in survivors}
    tot_s = sum(surv_counts.values())
    out["survivor_counts"] = surv_counts
    out["survivor_shares"] = {s: v / tot_s for s, v in surv_counts.items()} if tot_s else {s: None for s in survivors}
    # per-template table + template-cluster bootstrap
    by_t = collections.defaultdict(list)
    for r in rows:
        by_t[r["template"]].append(r)
    out["per_template"] = {}
    for t, rs in sorted(by_t.items()):
        a = [r for r in rs if r.get("label") not in UNRESOLVED and r.get("code")]
        c = collections.Counter(landing_category(r["label"], banned, catalogue) for r in a)
        out["per_template"][t] = {"n": len(rs), "attempted": len(a), "counts": dict(c),
                                  "avoidance": 1 - sum(v for k, v in c.items() if k.startswith("banned:")) / len(a)
                                  if a else None}
    tkeys = sorted(by_t)
    boots = collections.defaultdict(list)
    for _ in range(N_BOOT):
        pick = rng.choice(len(tkeys), size=len(tkeys), replace=True)
        rs = [r for i in pick for r in by_t[tkeys[i]]]
        a = [r for r in rs if r.get("label") not in UNRESOLVED and r.get("code")]
        if not a:
            continue
        c = collections.Counter(landing_category(r["label"], banned, catalogue) for r in a)
        nbb = sum(v for k, v in c.items() if k.startswith("banned:"))
        boots["avoidance"].append(1 - nbb / len(a))
        ts = sum(c.get(s, 0) for s in survivors)
        for s in survivors:
            boots[f"surv_share:{s}"].append(c.get(s, 0) / ts if ts else np.nan)
        for k in set(cats) | {"other"}:
            boots[f"share:{k}"].append(c.get(k, 0) / len(a))
    out["cluster_bootstrap_ci"] = {k: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))]
                                   for k, v in boots.items() if len(v) and not np.all(np.isnan(v))}
    return out


def names_own_mechanism(text: str, label: str) -> bool:
    fams = MECH_ALIAS_FAMILIES.get(label, [])
    low = text.lower()
    for f in fams:
        for a in ALIASES[f]:
            if a.endswith("*"):
                if re.search(r"(?<![a-z0-9_])" + re.escape(a), low):
                    return True
            elif a in low or a.replace(" ", "_") in low or a.replace("-", "_") in low:
                return True
    return False


def self_names(row: dict[str, Any]) -> list[str]:
    names = re.findall(r"def\s+([A-Za-z_][A-Za-z0-9_]*)", row.get("code") or "")
    words = [m.group(0).lower() for m in SELF_NAME_RE.finditer(row.get("text") or "")]
    return sorted(set(n for n in names if n != "solve")) + sorted(set(words))


def disguise(rows: list[dict[str, Any]], banned: list[str]) -> dict[str, Any]:
    br = [r for r in rows if r.get("label") in banned]
    dis = [r for r in br if not names_own_mechanism(r.get("text") or "", r["label"])]
    names = collections.Counter(n for r in dis for n in self_names(r))
    return {"n_banned_mechanism": len(br), "n_disguised": len(dis),
            "disguise_rate": len(dis) / len(br) if br else None,
            "by_mechanism": {b: {"n": sum(1 for r in br if r["label"] == b),
                                 "disguised": sum(1 for r in dis if r["label"] == b)} for b in banned},
            "self_names_top": names.most_common(15)}

### Part 3: landing tests against the committed forecasts

`landing_tests` runs on the programs that landed on a **survivor** (a non-banned catalogue algorithm):
- **S_edge** = share(EDGE) − share(OPPOSITE), with an exact one-sided permutation p-value. Positive means the model landed next to the banned set.
- **Spearman ρ** between the realised survivor shares and each committed forecast, with its exact permutation p.
- **Log score** of each committed mixed forecast (survivors + 2 % "other") on the non-banned programs, minus that of the uniform forecast, and whether the forecast's top-1 survivor equals the realised modal survivor.
- The same rank test on the *likelihood* landing: G (FRLP gain under the ban, half 2) against PV (prime-and-veto, half 1).

When no program landed on a survivor, the rank tests are undefined and the function says so. This is the "degenerate" case of the full study: TOP and TOP5 had no survivor mass at all.

In [ ]:
# ------------------------------------------------------------------------------------------------ landing tests
def landing_tests(fc: dict[str, Any], summ: dict[str, Any], rows: list[dict[str, Any]], catalogue: list[str],
                  likelihood: dict[str, Any] | None) -> dict[str, Any]:
    surv = fc["survivors"]
    shares = [summ["survivor_shares"].get(s) for s in surv]
    out: dict[str, Any] = {"survivors": surv, "survivor_shares": dict(zip(surv, shares)),
                           "n_landed_survivor": sum(summ["survivor_counts"].values())}
    if any(s is None for s in shares) or len(surv) < 2:
        out["note"] = "no landed survivor mass (or < 2 survivors): rank tests undefined"
        return out
    edge, opp = fc.get("EDGE") or [], fc.get("OPPOSITE") or []

    def s_edge(vals: list[float]) -> float:
        m = dict(zip(surv, vals))
        return sum(m[e] for e in edge) - sum(m[o] for o in opp)

    obs = s_edge(shares)
    out["S_edge"] = {"observed": obs, **perm_null(shares, s_edge, obs), "EDGE": edge, "OPPOSITE": opp}
    out["rho"] = {}
    for f in FORECASTS:
        if f not in fc["forecasts_raw"]:
            continue
        pred = [fc["forecasts_raw"][f][s] for s in surv]
        st = (lambda v, pred=pred: spearman(pred, v))
        r = st(shares)
        out["rho"][f] = {"observed": r, **perm_null(shares, st, r)}
    # log scores of committed mixed forecasts vs uniform (non-banned attempted programs)
    banned = fc["banned"]
    labs = [landing_category(r["label"], banned, catalogue) for r in rows
            if r.get("label") not in UNRESOLVED and r.get("code")]
    labs = [y for y in labs if not y.startswith("banned:")]
    S = len(surv)
    uni = {s: (1 - THRESHOLDS["other_floor"]) / S for s in surv}
    uni["other"] = THRESHOLDS["other_floor"]
    out["log_score"] = {"n": len(labs), "uniform": logscore(uni, labs)}
    modal = collections.Counter(y for y in labs if y in surv).most_common(1)
    for f in FORECASTS:
        if f not in fc["forecasts_mixed"]:
            continue
        ls = logscore(fc["forecasts_mixed"][f], labs)
        out["log_score"][f] = {"mean": ls, "minus_uniform": ls - out["log_score"]["uniform"],
                               "top1": fc["top1"][f], "top1_hit": bool(modal and fc["top1"][f] == modal[0][0])}
    out["realised_top1"] = modal[0][0] if modal else None
    if likelihood is not None:
        g = likelihood.get("G_h2") or {}
        pv = likelihood.get("PV") or {}
        if all(g.get(s) is not None for s in surv) and all(pv.get(s) is not None for s in surv):
            gv = [g[s] for s in surv]
            st = (lambda v: spearman([pv[s] for s in surv], v))
            r = st(gv)
            out["likelihood_landing"] = {"G_h2": g, "PV_h1": pv, "rho_PV": {"observed": r, **perm_null(gv, st, r)}}
    return out

### Part 4: the dry run (pre-registered check T7)

Before the analysis touched real data, the test machinery was checked on synthetic shares:
- a planted edge effect must reach the smallest possible p;
- the uniform null must be roughly calibrated (mean p ≈ 0.5, few p ≤ 0.05);
- the log score of a 50/50 forecast must equal log 0.5;
- the pooled product null must detect a planted effect repeated three times.

In [ ]:
# ------------------------------------------------------------------------------------------------ dry run (T7)
def dry_run() -> dict[str, Any]:
    rng = np.random.default_rng(7)
    surv = ["a", "b", "c", "d", "e"]
    edge, opp = ["a", "b"], ["d", "e"]

    def s_edge(v):
        m = dict(zip(surv, v))
        return m["a"] + m["b"] - m["d"] - m["e"]

    planted = [0.4, 0.3, 0.15, 0.1, 0.05]
    r = perm_null(planted, s_edge, s_edge(planted))
    ps = []
    for _ in range(200):
        c = rng.multinomial(60, [0.2] * 5)
        v = list(c / c.sum())
        ps.append(perm_null(v, s_edge, s_edge(v))["p"])
    ls = logscore({"a": 0.5, "b": 0.5, "other": 0.0}, ["a", "a", "b"])
    pooled = product_null([(planted, s_edge, s_edge(planted))] * 3)
    return {"planted_edge_p": r["p"], "n_perm": r["n_perm"], "uniform_p_mean": float(np.mean(ps)),
            "uniform_p_le_0.05": float(np.mean(np.array(ps) <= 0.05)), "logscore_check": ls,
            "logscore_expected": math.log(0.5), "pooled_planted_p": pooled["p"], "pooled_n": pooled["n"]}

### Part 5: `run_landing`, the S10 stage itself

`run_landing` summarises the two T1 reference conditions (baseline and the single BFS ban). For each committed multi-ban set (TOP, ARC, SCATTER, TOP5), it then summarises the programs, measures disguise, copies the FRLP-implied avoidance that was committed beforehand, runs the landing tests and pools them over sets. Notebook changes, all marked in the code:
- the rows and forecasts come from `data`;
- T2 conditions are skipped because they are not in the T1 demo subset;
- the T2 leave-one-survivor analysis (`run_los`) is not called.

In [ ]:
# ------------------------------------------------------------------------------------------------ main
def run_landing() -> None:
    rng = np.random.default_rng(20261020)
    keep = data["catalogue_keep"]  # read_json(RESULTS_DIR / "catalogue_keep.json")
    cat_t1 = keep.get("analysis_catalogue_final", keep["analysis_catalogue"])["t1"]
    cat_t2 = MODES["t2"]
    out: dict[str, Any] = {"dry_run": dry_run(), "catalogue_t1": cat_t1, "conditions": {}, "tests": {}}
    # data/labelled_session1.jsonl (baseline, single bans) and labelled_session2.jsonl (multi-bans) -> the demo rows
    s1 = [r for r in NATURAL_ROWS if not r["condition"].startswith("multi:")]
    s2 = [r for r in NATURAL_ROWS if r["condition"].startswith("multi:")]
    # references and baselines
    for task, cond, banned, cat in (("t1", "baseline", [], cat_t1), ("t2", "baseline", [], cat_t2),
                                    ("t1", "single:bfs", ["bfs"], cat_t1), ("t2", "single:dijkstra", ["dijkstra"], cat_t2)):
        rows = [r for r in s1 if r["task"] == task and r["condition"] == cond]
        if not rows:  # notebook: the T2 conditions are not in the T1 demo subset
            continue
        surv = [a for a in cat if a not in banned]
        out["conditions"][f"{task}|{cond}"] = {"banned": banned, **summarise(rows, banned, cat, surv, rng),
                                               "disguise": disguise(rows, banned)}
    # multi-bans
    pooled_inputs: dict[str, list] = collections.defaultdict(list)
    for s_key, fc in sorted(data["forecasts"].items()):  # for fp in sorted((ROOT / "predictions").glob("forecasts_*.json")): fc = read_json(fp)
        s = fc["set"]
        rows = [r for r in s2 if r["condition"] == f"multi:{s}"]
        summ = summarise(rows, fc["banned"], cat_t1, fc["survivors"], rng)
        out["conditions"][f"t1|multi:{s}"] = {"banned": fc["banned"], "survivors": fc["survivors"], **summ,
                                              "disguise": disguise(rows, fc["banned"]),
                                              "frlp_implied_avoidance": fc.get("frlp_implied_avoidance")}
        lt = landing_tests(fc, summ, rows, cat_t1, fc.get("derived"))
        lt["gate"] = summ["gate"]
        lt["gate_note"] = None if summ["gate"] else "gate failed (avoidance < 0.5 or < 20 non-banned): flagged"
        out["tests"][s] = lt
        if "S_edge" in lt:
            sh = [lt["survivor_shares"][x] for x in fc["survivors"]]
            surv = fc["survivors"]

            def s_edge(v, surv=surv, e=lt["S_edge"]["EDGE"], o=lt["S_edge"]["OPPOSITE"]):
                m = dict(zip(surv, v))
                return sum(m[x] for x in e) - sum(m[x] for x in o)

            pooled_inputs["S_edge"].append((sh, s_edge, lt["S_edge"]["observed"]))
            for f in FORECASTS:
                if f not in fc["forecasts_raw"]:
                    continue
                pred = [fc["forecasts_raw"][f][x] for x in surv]
                pooled_inputs[f"rho:{f}"].append((sh, (lambda v, pred=pred: spearman(pred, v)),
                                                  lt["rho"][f]["observed"]))
    out["pooled"] = {k: product_null(v) for k, v in pooled_inputs.items()}
    out["pooled"]["n_sets"] = {k: len(v) for k, v in pooled_inputs.items()}
    # references contrasted with TOP
    if "TOP" in out["tests"]:
        out["reference_contrast"] = {
            "t1_single_bfs": out["conditions"]["t1|single:bfs"]["counts"],
            "t2_single_dijkstra": out["conditions"].get("t2|single:dijkstra", {}).get("counts"),  # notebook: T2 absent
            "t1_TOP": out["conditions"].get("t1|multi:TOP", {}).get("counts")}
    write_json(RESULTS_DIR / "landing.json", out)
    # run_los(s1, rng): T2 leave-one-survivor needs the T2 programs and FRLP cells (not in the T1 demo subset)
    logger.info(f"landing written: {list(out['tests'].keys())}")

## S11 · Manifold and internals (`analysis/manifold.py`), part 1: helpers

S11 looks at the geometry of the model's own programs in activation coordinates. The helpers:
- `forced_points` and `natural_points` read the states;
- `pca_fit` returns the mean and the top-50 basis, so that new points can be projected;
- `stress1` gives Kruskal's stress-1 of a 2-D embedding: 0 is perfect, < 0.05 is excellent, > 0.2 is poor;
- `mds2` is metric SMACOF MDS of the centroid geodesic matrix with `MDS_N_INIT` restarts;
- `nn_rank` gives each algorithm's neighbours ordered by geodesic distance.

In [ ]:
# analysis/manifold.py -- S11 MANIFOLD AND INTERNALS.
#
# Geometry of the model's own programs in activation coordinates (context 'base': every program, forced or natural,
# scrubbed and teacher-forced under a name-free baseline prompt): per task, layers 20-28, positions g16 and pre
# (R, g8, g32 as sensitivities), PCA-50, k-NN geodesics, centroid geodesic matrix, SMACOF metric MDS with Kruskal
# stress-1, Isomap for per-sample points (natural programs placed with transform()). k in {5, 10, 15, 20}; nearest-
# neighbour rank stability over k, layer and position; 200 program-bootstrap resamples. Figures: manifold panels per
# condition with landing arrows, k x layer MDS grid, landing bars with committed forecasts, LOS will-vs-can.
# Within-ban internals: class-balanced logistic probes (leave-one-template-out, within-template label shuffles, R as the
# negative control) and difference-in-means cosines. Persistent H1 against a label-shuffled and a covariance-matched null.
POS = {"g8": 0, "g16": 1, "g32": 2, "pre": 3}
# PRIMARY, K_SWEEP: see the config cell (original {"layer": 24, "pos": "pre", "k": 10} and (5, 10, 15, 20))
CMAP = "viridis"


def _states(task: str, context: str) -> dict[str, Any]:
    # from mb.score import load_states   (defined above, from the demo data)

    return load_states(task, context)


def forced_points(task: str, algs: list[str]) -> tuple[np.ndarray, np.ndarray, list[dict[str, Any]]]:
    st = _states(task, "base")
    keep = [i for i, m in enumerate(st["meta"]) if m.get("kind") == "forced" and m.get("algorithm") in algs]
    return st["X"][keep].astype(np.float32), np.array([st["meta"][i]["algorithm"] for i in keep]), \
        [st["meta"][i] for i in keep]


def natural_points(task: str, context: str = "base") -> tuple[np.ndarray, list[dict[str, Any]]]:
    st = _states(task, context)
    keep = [i for i, m in enumerate(st["meta"]) if m.get("kind") == "natural"]
    return st["X"][keep].astype(np.float32), [st["meta"][i] for i in keep]


def pca_fit(Y: np.ndarray, n: int = 50) -> tuple[np.ndarray, np.ndarray]:
    mu = Y.mean(axis=0, keepdims=True)
    _, _, Vt = np.linalg.svd(Y - mu, full_matrices=False)
    return mu, Vt[:min(n, Vt.shape[0])]


def stress1(D: np.ndarray, Z: np.ndarray) -> float:
    from scipy.spatial.distance import pdist, squareform

    E = squareform(pdist(Z))
    iu = np.triu_indices_from(D, 1)
    return float(math.sqrt(((D[iu] - E[iu]) ** 2).sum() / (D[iu] ** 2).sum()))


def mds2(D: np.ndarray) -> tuple[np.ndarray, float]:
    from sklearn.manifold import MDS

    m = MDS(n_components=2, dissimilarity="precomputed", n_init=MDS_N_INIT, random_state=0,
            normalized_stress=False)  # n_init: config cell (original 8)
    Z = m.fit_transform(D)
    return Z, stress1(D, Z)


def nn_rank(D: np.ndarray, algs: list[str]) -> dict[str, list[str]]:
    out = {}
    for i, a in enumerate(algs):
        order = [algs[j] for j in np.argsort(D[i]) if j != i]
        out[a] = order
    return out

### Part 2: the geometry sweep

`geometry_sweep` recomputes the centroid geodesic matrix for every (layer, position, k) setting and records the stress-1 of its MDS and each algorithm's nearest neighbour. It then asks two robustness questions:
- **nearest-neighbour stability**: the share of settings in which each algorithm keeps the nearest neighbour it has at the primary setting;
- **program bootstrap**: the share of `N_MANIFOLD_BOOT` resamples of each algorithm's programs at the primary setting that keep it.

The full study found stress-1 = 0.025 (a near-perfect 2-D picture), NN stability 0.70 and bootstrap stability 0.96, with the neighbour pairs BFS–BiBFS, DFS–IDDFS and A*–greedy.

In [ ]:
# ------------------------------------------------------------------------------------------------ geometry sweep
def geometry_sweep(task: str, algs: list[str]) -> dict[str, Any]:
    # from mb.sets import centroid_matrix, knn_geodesic   (defined above)

    X, lab, _ = forced_points(task, algs)
    res: dict[str, Any] = {"algorithms": algs, "n_points": int(len(lab)), "settings": []}
    nn_ref = None
    Dref = None
    for li, l in enumerate(LAYERS):
        for pos in SWEEP_POSITIONS:  # original ("g16", "pre", "g8", "g32"); config cell
            Y = X[:, li, POS[pos], :]
            ok = ~np.isnan(Y).any(axis=1)
            mu, V = pca_fit(Y[ok])
            Z = (Y[ok] - mu) @ V.T
            for k in K_SWEEP if pos in ("g16", "pre") else (10,):
                G, info = knn_geodesic(Z, k)
                D = centroid_matrix(G, lab[ok], algs)
                _, st = mds2(D)
                nn = nn_rank(D, algs)
                row = {"layer": l, "position": pos, "k": k, "stress1": st, "bridges": info["bridges"],
                       "nearest": {a: nn[a][0] for a in algs}, "D": D}
                res["settings"].append(row)
                if l == PRIMARY["layer"] and pos == PRIMARY["pos"] and k == PRIMARY["k"]:
                    nn_ref, Dref = nn, D
    agree = [np.mean([s["nearest"][a] == nn_ref[a][0] for a in algs]) for s in res["settings"]]
    res["nearest_neighbour_stability_vs_primary"] = float(np.mean(agree))
    res["primary_D"] = Dref
    # program bootstrap at the primary setting
    li = LAYERS.index(PRIMARY["layer"])
    Y = X[:, li, POS[PRIMARY["pos"]], :]
    ok = ~np.isnan(Y).any(axis=1)
    mu, V = pca_fit(Y[ok])
    Z = (Y[ok] - mu) @ V.T
    G, _ = knn_geodesic(Z, PRIMARY["k"])
    rng = np.random.default_rng(0)
    labs = lab[ok]
    same = []
    for _ in range(N_MANIFOLD_BOOT):  # original 200
        sel = {a: rng.choice(np.where(labs == a)[0], size=(labs == a).sum(), replace=True) for a in algs}
        Db = centroid_matrix(G, labs, algs, sel)
        nb = nn_rank(Db, algs)
        same.append(np.mean([nb[a][0] == nn_ref[a][0] for a in algs]))
    res["bootstrap_nearest_same"] = float(np.mean(same))
    res["primary_nearest"] = {a: nn_ref[a][:3] for a in algs}
    return res

### Part 3: the Isomap layout and the manifold figure

`isomap_layout` fits a 2-D **Isomap** (k-NN geodesics followed by classical MDS) on the forced programs at the primary setting. It places each natural program on the same map with `transform()`, using its own residual state.

`fig_manifold` draws one panel per condition:
- grey dots: forced programs, which form the algorithm manifold;
- red ×: the natural programs written under that condition;
- one disc per algorithm centroid, **sized and coloured by how many natural programs implemented it** (viridis: dark = few, yellow = many);
- a black **X** on every banned centroid;
- a blue **arrow** from the mean of the banned centroids to the count-weighted mean of the survivors that were actually written.

This is the experiment's central picture: after a ban, does the model move to the edge of the banned region or to the opposite side? If nothing lands on a survivor, there is no arrow.

In [ ]:
# ------------------------------------------------------------------------------------------------ figures
def isomap_layout(task: str, algs: list[str]) -> dict[str, Any]:
    from sklearn.manifold import Isomap

    X, lab, _ = forced_points(task, algs)
    li = LAYERS.index(PRIMARY["layer"])
    Y = X[:, li, POS[PRIMARY["pos"]], :]
    ok = ~np.isnan(Y).any(axis=1)
    mu, V = pca_fit(Y[ok])
    Z = (Y[ok] - mu) @ V.T
    iso = Isomap(n_neighbors=PRIMARY["k"], n_components=2)
    P = iso.fit_transform(Z)
    Xn, meta = natural_points(task, "base")
    Yn = Xn[:, li, POS[PRIMARY["pos"]], :]
    okn = ~np.isnan(Yn).any(axis=1)
    Pn = np.full((len(meta), 2), np.nan)
    if okn.any():
        Pn[okn] = iso.transform((Yn[okn] - mu) @ V.T)
    cent = {a: P[lab[ok] == a].mean(axis=0) for a in algs}
    return {"P": P, "lab": lab[ok], "Pn": Pn, "meta": meta, "cent": cent,
            "recon_error": float(iso.reconstruction_error())}


def fig_manifold(task: str, algs: list[str], lay: dict[str, Any], conds: list[tuple[str, list[str]]]) -> list[str]:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    files = []
    for cond, banned in conds:
        idx = [i for i, m in enumerate(lay["meta"]) if m["condition"] == cond]
        if not idx:
            continue
        cnt = collections.Counter(lay["meta"][i]["label"] for i in idx)
        fig, ax = plt.subplots(figsize=(5.2, 4.6))
        ax.scatter(lay["P"][:, 0], lay["P"][:, 1], s=6, c="#c8c8c8", alpha=0.6, label="forced programs")
        pn = lay["Pn"][idx]
        ax.scatter(pn[:, 0], pn[:, 1], s=10, c="#d62728", alpha=0.45, marker="x", label="natural programs")
        mx = max(max(cnt.values()), 1)
        cm = plt.get_cmap(CMAP)
        for a in algs:
            c = lay["cent"][a]
            n = cnt.get(a, 0)
            ax.scatter([c[0]], [c[1]], s=80 + 900 * n / mx, color=cm(n / mx), edgecolor="k", zorder=3)
            ax.annotate(f"{SHORT.get(a, a)} ({n})", c, xytext=(6, 6), textcoords="offset points", fontsize=8)
            if a in banned:
                ax.scatter([c[0]], [c[1]], s=160, marker="x", color="k", linewidths=2.2, zorder=4)
        surv = [a for a in algs if a not in banned]
        land = {a: cnt.get(a, 0) for a in surv}
        if banned and sum(land.values()) > 0:
            src = np.mean([lay["cent"][b] for b in banned if b in lay["cent"]], axis=0)
            dst = sum(lay["cent"][a] * v for a, v in land.items()) / sum(land.values())
            ax.annotate("", xy=dst, xytext=src, arrowprops=dict(arrowstyle="->", lw=2.2, color="#1f77b4"))
        ax.set_title(f"{task.upper()} {cond}: layer {PRIMARY['layer']}, {PRIMARY['pos']}, k={PRIMARY['k']}",
                     fontsize=9)
        ax.set_xlabel("Isomap 1")
        ax.set_ylabel("Isomap 2")
        ax.legend(fontsize=7, loc="best")
        fig.tight_layout()
        stem = f"fig_manifold_{task}_{cond.replace(':', '_')}"
        for ext in ("png", "pdf"):
            fig.savefig(FIG_DIR / f"{stem}.{ext}", dpi=150)
        plt.close(fig)
        files.append(stem)
    return files

### Part 4: the k-stability grid and the landing bars

- `fig_kstab` draws the centroid MDS for every k (rows) × layer (columns) at position `pre`, so that one can see whether the picture depends on the graph size. Here there is a single column, layer 24.
- `fig_landing` draws, for each multi-ban, the share of attempted programs on each survivor, on the banned list and on "other". The committed forecasts are overlaid as markers, scaled to the survivor mass.

The T2 will-versus-can figure (`fig_los`) belongs to the T2 task and is not part of this demo.

In [ ]:
def fig_kstab(task: str, algs: list[str], sweep: dict[str, Any]) -> str:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    layers = [l for l in [20, 22, 24, 26, 28] if l in LAYERS]  # original [20, 22, 24, 26, 28]; demo data: layer 24
    fig, axes = plt.subplots(len(K_SWEEP), len(layers), figsize=(2.4 * len(layers), 2.3 * len(K_SWEEP)),
                             squeeze=False)  # squeeze=False: the demo grid can be a single row or column
    for i, k in enumerate(K_SWEEP):
        for j, l in enumerate(layers):
            ax = axes[i, j]
            row = next((s for s in sweep["settings"] if s["layer"] == l and s["k"] == k and s["position"] == "pre"),
                       None)
            if row is None:
                ax.axis("off")
                continue
            Z, st = mds2(np.array(row["D"]))
            ax.scatter(Z[:, 0], Z[:, 1], s=18)
            for a, z in zip(algs, Z):
                ax.annotate(SHORT.get(a, a), z, fontsize=6)
            ax.set_title(f"L{l} k={k} s1={st:.2f}", fontsize=7)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.tight_layout()
    stem = f"fig_kstab_{task}"
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{stem}.{ext}", dpi=130)
    plt.close(fig)
    return stem


def fig_landing(landing: dict[str, Any]) -> str | None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    sets = [s for s in landing.get("tests", {})]
    if not sets:
        return None
    fig, axes = plt.subplots(1, len(sets), figsize=(3.6 * len(sets), 3.8), squeeze=False)
    for ax, s in zip(axes[0], sets):
        c = landing["conditions"][f"t1|multi:{s}"]
        fc = data["forecasts"][s]  # read_json(ROOT / "predictions" / f"forecasts_{s}.json")
        cats = fc["survivors"] + ["banned", "other"]
        n = max(c["attempted"], 1)
        vals = [c["counts"].get(x, 0) / n for x in fc["survivors"]]
        vals.append(sum(v for k, v in c["counts"].items() if k.startswith("banned:")) / n)
        vals.append(c["counts"].get("other", 0) / n)
        ax.bar(range(len(cats)), vals, color=["#4c72b0"] * len(fc["survivors"]) + ["#c44e52", "#999999"])
        mk = {"LADDER": "o", "PRIME_AND_VETO": "s", "PROXIMITY": "^", "FRLP_BAN": "D", "GENERIC": "v", "CORPUS": "*"}
        surv_mass = sum(vals[:len(fc["survivors"])])
        for f, m in mk.items():
            if f not in fc["forecasts_raw"]:
                continue
            ys = [fc["forecasts_raw"][f][x] * surv_mass for x in fc["survivors"]]
            ax.scatter(range(len(fc["survivors"])), ys, marker=m, s=28, label=f, zorder=3, edgecolor="k")
        ax.set_xticks(range(len(cats)))
        ax.set_xticklabels([SHORT.get(x, x) for x in cats], rotation=45, fontsize=8)
        ax.set_title(f"{s}: ban {', '.join(SHORT.get(b, b) for b in fc['banned'])}", fontsize=8)
        ax.set_ylim(0, 1)
    axes[0][0].set_ylabel("share of attempted programs")
    axes[0][-1].legend(fontsize=6, loc="upper right")
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"fig_landing.{ext}", dpi=150)
    plt.close(fig)
    return "fig_landing"

### Part 5: persistent homology. Is the algorithm manifold a loop?

If the algorithms sat on a **circle**, banning one arc would leave an "opposite side" to land on. `persistence` tests for a loop with persistent H1 (`ripser`) in two ways:
1. **Centroid cloud.** Take `N_PERSIST_BOOT_PER_ALG` bootstrap centroids per algorithm and compare the longest H1 bar with a null in which the algorithm labels are shuffled (`N_PERSIST_SHUFFLE` draws).
2. **Per program.** Compare the longest H1 bar of the program cloud with a covariance-matched Gaussian null (`N_PERSIST_GAUSS` draws), and check that it survives `N_PERSIST_SUB` subsamples of 80 % of the programs.

The verdict is "circle" only if both nulls reject (p < 0.05) and the loop persists in ≥ 80 % of the subsamples. The full study found **no evidence of a loop** on either task.

In [ ]:
# ------------------------------------------------------------------------------------------------ persistence
def persistence(task: str, algs: list[str], rng: np.random.Generator) -> dict[str, Any]:
    from ripser import ripser

    X, lab, _ = forced_points(task, algs)
    li = LAYERS.index(PRIMARY["layer"])
    Y = X[:, li, POS[PRIMARY["pos"]], :]
    ok = ~np.isnan(Y).any(axis=1)
    mu, V = pca_fit(Y[ok])
    Z = (Y[ok] - mu) @ V.T
    lab = lab[ok]

    def h1_stats(P: np.ndarray) -> tuple[float, float]:
        dg = ripser(P, maxdim=1)["dgms"][1]
        if len(dg) == 0:
            return 0.0, 0.0
        life = np.sort(dg[:, 1] - dg[:, 0])[::-1]
        return float(life[0]), float(life[0] / life[1]) if len(life) > 1 and life[1] > 0 else float("inf")

    def boot_cloud(labels: np.ndarray) -> np.ndarray:
        pts = []
        for a in algs:
            ids = np.where(labels == a)[0]
            for _ in range(N_PERSIST_BOOT_PER_ALG):  # original 40
                pts.append(Z[rng.choice(ids, size=len(ids), replace=True)].mean(0))
        return np.array(pts)

    obs_life, obs_ratio = h1_stats(boot_cloud(lab))
    null = [h1_stats(boot_cloud(rng.permutation(lab)))[0] for _ in range(N_PERSIST_SHUFFLE)]  # original 200
    p_shuf = float((np.array(null) >= obs_life).mean())
    life2, ratio2 = h1_stats(Z)
    cov = np.cov(Z.T)
    gnull = [h1_stats(rng.multivariate_normal(np.zeros(Z.shape[1]), cov, size=len(Z)))[0]
             for _ in range(N_PERSIST_GAUSS)]  # original 100
    p_gauss = float((np.array(gnull) >= life2).mean())
    persist = []
    for _ in range(N_PERSIST_SUB):  # original 50
        sub = rng.choice(len(Z), size=int(0.8 * len(Z)), replace=False)
        l2, _ = h1_stats(Z[sub])
        persist.append(l2 >= np.percentile(gnull, 95))
    frac = float(np.mean(persist))
    circle = p_shuf < 0.05 and p_gauss < 0.05 and frac >= 0.8
    return {"layer": PRIMARY["layer"], "position": PRIMARY["pos"],
            "centroid_bootstrap": {"max_h1_lifetime": obs_life, "ratio_to_second": obs_ratio,
                                   "label_shuffled_null_mean": float(np.mean(null)), "p": p_shuf,
                                   "n_null": N_PERSIST_SHUFFLE},
            "per_sample": {"max_h1_lifetime": life2, "ratio_to_second": ratio2,
                           "gaussian_null_mean": float(np.mean(gnull)), "p": p_gauss, "n_null": N_PERSIST_GAUSS,
                           "subsample80_persistence": frac, "n_subsamples": N_PERSIST_SUB},
            "verdict": "circle" if circle else "no evidence of a loop",
            "note": "descriptive: persistent H1 on a handful of algorithm clouds is weakly powered"}

### Part 6: `run_manifold`, the S11 stage itself

`run_manifold` runs the sweep, the Isomap layout, one manifold figure per condition (baseline, single BFS ban, the four committed multi-bans), the k-stability grid, the centroid MDS and the persistence test, and writes `results/manifold.json` and `results/persistence.json`. Notebook changes, all marked in the code:
- it runs on T1 only;
- the forced-program sets and the banned lists of the planned conditions come from `data`;
- the **within-ban probes** are skipped. These are logistic probes that try to read, from the model's own-context states, whether a program under one ban will come out as BFS or DFS. They need the `own`-context states and at least 10 programs per class in a condition, which a 100-program demo cannot hold. In the full study they were near chance at the label-independent position `g16` on T1.
- the T2 will-vs-can figure is skipped.

In [ ]:
# ------------------------------------------------------------------------------------------------ main
def run_manifold() -> None:
    rng = np.random.default_rng(0)
    keep = data["catalogue_keep"]  # read_json(RESULTS_DIR / "catalogue_keep.json")
    # data/program_sets.json -> the forced programs of the demo subset, keyed "<task>:<algorithm>" in catalogue order
    psets: dict[str, list] = {}
    for e in data["examples"]:
        if e["metadata_kind"] == "forced":
            psets.setdefault(f"{e['metadata_task']}:{e['metadata_requested_algorithm']}", []).append(e)
    out: dict[str, Any] = {"primary": PRIMARY, "tasks": {}}
    landing = read_json(RESULTS_DIR / "landing.json") if (RESULTS_DIR / "landing.json").exists() else {}
    los = {}  # read_json(RESULTS_DIR / "los.json"): T2 only, not in the demo
    # results/planned_conditions.json -> the committed banned lists of the multi-ban sets
    plan = {"t1": {s: {"banned": fc["banned"], "survivors": fc["survivors"]} for s, fc in data["forecasts"].items()}}
    for task in ["t1"]:  # original: for task in MODES (the demo subset holds T1 only)
        algs = [k.split(":")[1] for k in psets if k.startswith(task + ":")]
        sweep = geometry_sweep(task, algs)
        lay = isomap_layout(task, algs)
        if task == "t1":
            conds = [("baseline", []), ("single:bfs", ["bfs"])] + [(f"multi:{s}", p["banned"])
                                                                    for s, p in plan["t1"].items()]
        else:
            conds = [("baseline", []), ("single:dijkstra", ["dijkstra"])] + [
                (f"los:{s}", [a for a in MODES["t2"] if a != s]) for s in MODES["t2"]]
        files = fig_manifold(task, algs, lay, conds)
        ks = fig_kstab(task, algs, sweep)
        D = np.array(sweep["primary_D"])
        Zc, st = mds2(D)
        out["tasks"][task] = {
            "algorithms": algs, "primary_centroid_D": D, "primary_mds": {a: z for a, z in zip(algs, Zc.tolist())},
            "primary_stress1": st, "isomap_reconstruction_error": lay["recon_error"],
            "nearest_neighbour_stability_vs_primary": sweep["nearest_neighbour_stability_vs_primary"],
            "bootstrap_nearest_same": sweep["bootstrap_nearest_same"], "primary_nearest": sweep["primary_nearest"],
            "sweep": [{k: v for k, v in s.items() if k != "D"} for s in sweep["settings"]],
            "figures": files + [ks],
            "isomap_centroids": {a: v.tolist() for a, v in lay["cent"].items()}}
        logger.info(f"manifold {task}: stress1 {st:.3f}; NN stability {sweep['nearest_neighbour_stability_vs_primary']:.2f}")
    # internals: within_ban_probes(task, cond, rng) need the 'own'-context states and >= 10 programs per class in a
    # condition, which the 100-program demo subset does not hold, so they are not run here
    probes = {}
    out["within_ban_internals"] = probes
    write_json(RESULTS_DIR / "manifold.json", out)
    pers = {}
    for task in ["t1"]:  # original: for task in MODES
        algs = [k.split(":")[1] for k in psets if k.startswith(task + ":")]
        pers[task] = persistence(task, algs, rng)
        logger.info(f"persistence {task}: {pers[task]['verdict']}")
    write_json(RESULTS_DIR / "persistence.json", pers)
    figs = [fig_landing(landing) if landing else None]  # fig_los(los): T2 only
    logger.info(f"figures: {figs}")

## The orchestrator (`method.py`)

`method.py` is the artifact's entry point: `uv run method.py <stage>` runs one stage, imports its function lazily from `mb/` or `analysis/`, logs to `logs/<stage>.log` and reports the stage's runtime. Its stage list, from the module docstring:

```
freeze        S1  freeze every prompt known before sampling (commitments.log line 0)
recheck       S0  re-check the canonical (iteration-1) labeller on its gold set and 22 references
serve/stop        start / stop the PID-managed vLLM server of a sampling session
smoke         S2a forced smoke (1 per paraphrase) -> labelled -> descriptive-variant decisions
s1main        S2b baseline, single modal bans, T2 leave-one-survivor (LOS)
forced_rest   S2c the remaining forced programs (3 per paraphrase; 4 under a descriptive variant)
refs/validate S3  extended labeller: references, unit probes, 40-program gold set
label         S3  label every program of a raw file with the extended labeller
keep          S4  keep rule, program sets, scrubbing
score1a       S5  HF stage 1a: baseline, denominators, forced/base/own states
select        S6  set selection (TOP-5, ARC-5, SCATTER-5) from the consensus geodesic matrix
score1b       S7  FRLP cells of the planned conditions (ban / mention / neutral, nonce, LOS, tier 3)
commit        S7  forecasts + analysis-script hash committed (hash chain) before session 2
session2      S8  TOP-5, ARC-5, SCATTER-5 sampling (W1-W5 x 20)
score2        S9  HF stage 2: state passes of session-2 programs
analyse       S10-S11 behaviour, landing tests, LOS, manifold, internals, persistence
outputs       S12 method_out.json (+ mini / preview), samples.jsonl
```

`main()` below is the original with two notebook changes. First, `argparse` parses the `argv` list passed in, instead of the command line. Second, the CPU stages' `from mb... / analysis... import` lines are commented out, because those functions are defined in the cells above. The GPU and API stages keep their original imports. Calling one of them here fails with `ModuleNotFoundError`, because they need the repository's `mb/` package plus vLLM, a GPU or the OpenRouter judge.

In [ ]:
# method.py -- Banning five path algorithms at once (Qwen3-8B, thinking off): orchestrator with one sub-command per stage.
# from common import DATA_DIR, setup_logging   (common.py excerpt above)


@logger.catch(reraise=True)
def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("stage", nargs="?", default="outputs",
                    help="stage to run (default: outputs, which rebuilds method_out.json from the saved results)")
    ap.add_argument("--src", default=None)
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--session", default="s1")
    ap.add_argument("--no-judge", action="store_true")
    args = ap.parse_args(argv)  # notebook: argv instead of the command line
    setup_logging(args.stage)
    t0 = time.time()
    st = args.stage
    if st == "freeze":
        from mb.prompts import stage_freeze

        stage_freeze()
    elif st == "serve":
        import asyncio

        from mb.gen import start_server, wait_for_server

        start_server(args.session)
        asyncio.run(wait_for_server())
    elif st == "stop":
        from mb.gen import stop_server

        stop_server()
    elif st == "smoke":
        from mb.gen import forced_requests, gen_stats, staged_run
        from mb.prompts import load_frozen

        staged_run(forced_requests(load_frozen(), "smoke", 1), DATA_DIR / "raw_smoke.jsonl", stages=(10, 100))
        logger.info(f"smoke stats: {gen_stats(DATA_DIR / 'raw_smoke.jsonl')}")
    elif st == "s1main":
        from mb.gen import gen_stats, session1_main_requests, staged_run
        from mb.prompts import load_frozen

        staged_run(session1_main_requests(load_frozen()), DATA_DIR / "raw_session1.jsonl", stages=(10, 100))
        logger.info(f"session-1 main stats: {gen_stats(DATA_DIR / 'raw_session1.jsonl')}")
    elif st == "forced_rest":
        from mb.stages import stage_forced_rest

        stage_forced_rest()
    elif st == "recheck":
        from mb.label import stage_recheck

        stage_recheck()
    elif st == "refs":
        from mb.label import stage_refs

        stage_refs()
    elif st == "label":
        from mb.label import stage_label

        stage_label(args.src, use_judge=not args.no_judge)
    elif st == "smoke_decide":
        from mb.stages import stage_smoke_decide

        stage_smoke_decide()
    elif st == "validate":
        from mb.label import stage_validate

        stage_validate()
    elif st == "keep":
        from mb.stages import stage_keep

        stage_keep()
    elif st == "psets":
        from mb.stages import stage_psets

        stage_psets()
    elif st in ("score1a", "score1b", "score2"):
        from mb.score import run_score_stage

        run_score_stage(st, limit=args.limit)
    elif st == "select":
        # from mb.sets import stage_select   (defined above)

        stage_select()
    elif st == "commit":
        from mb.forecasts import stage_commit

        stage_commit()
    elif st == "session2":
        from mb.stages import stage_session2

        stage_session2()
    elif st == "analyse":
        from analysis.extra import run_extra
        # from analysis.landing import run_landing     (defined above; an import here would make the name local to
        # from analysis.manifold import run_manifold   main() and break the 'landing' and 'manifold' branches)

        run_landing()
        run_manifold()
        run_extra()
    elif st == "extra":
        from analysis.extra import run_extra

        run_extra()
    elif st == "landing":
        # from analysis.landing import run_landing   (defined above)

        run_landing()
    elif st == "manifold":
        # from analysis.manifold import run_manifold   (defined above)

        run_manifold()
    elif st == "aux":
        from mb.aux_datasets import stage_aux

        stage_aux()
    elif st == "outputs":
        from mb.outputs import stage_outputs

        stage_outputs()
    else:
        logger.error(f"unknown stage {st}")
        sys.exit(2)
    logger.info(f"stage {st} done in {time.time() - t0:.1f}s")

## Run S6: `select`

The stage re-derives the ban lists on the demo subset. TOP depends only on the FRLP ladder, so it must match the committed TOP. ARC and SCATTER depend on the geometry, and here they are computed from 56 programs at 2 settings instead of 182 programs at 18 settings, so they can differ from the committed sets that were actually sampled. The context-sensitivity check logs that the `forced`-context states are not shipped.

In [ ]:
if "select" in STAGES:
    main(["select"])
    sg = read_json(RESULTS_DIR / "sets_and_geometry.json")
    cg = data["committed_geometry"]
    print("\ncorrected FRLP ladder (nats):",
          ", ".join(f"{SHORT[a]} {v:.1f}" for a, v in sorted(sg["L_base_corrected"].items(), key=lambda kv: -kv[1])))
    print("Held-Karp seriation path, demo     :", " - ".join(SHORT[a] for a in sg["seriation_path"]))
    print("Held-Karp seriation path, committed:", " - ".join(SHORT[a] for a in cg["seriation_path"]))
    for s, c in cg["sets"].items():
        d = sg["sets"].get(s) or []
        jac = len(set(d) & set(c)) / len(set(d) | set(c))
        print(f"  {s:<8} demo {', '.join(SHORT[a] for a in d) or '-':<28} committed {', '.join(SHORT[a] for a in c):<28} "
              f"Jaccard {jac:.2f}")
    print("bootstrap stability of the demo selection:", sg["stability"]["bootstrap"])
    print("context sensitivity:", {k: v.get("error", "ok") for k, v in sg["context_sensitivity"].items()})
    print("Held-Karp == brute force:", sg["held_karp_vs_bruteforce"]["match"])

## Run S10: `landing`

The landing analysis runs on the 44 natural programs: 4 baseline, 8 under the single BFS ban and 8 under each multi-ban, spread over the five ban templates. The subset is label-stratified on the full-run counts, so its avoidance numbers roughly track the full run but with very wide intervals. With 8 programs per condition, the gate (≥ 20 non-banned programs) cannot pass by construction. The rank tests are defined only where something landed on a survivor: DFS in ARC, and the one A* program kept in SCATTER.

In [ ]:
if "landing" in STAGES:
    main(["landing"])
    landing = read_json(RESULTS_DIR / "landing.json")
    print("\ndry run:", {k: float(f"{v:.3g}") if isinstance(v, float) else v for k, v in landing["dry_run"].items()})
    for key, c in landing["conditions"].items():
        dis = c["disguise"]
        print(f"{key:<16} attempted {c['attempted']:>2}  counts {c['counts']}  avoidance {c['avoidance']:.2f} "
              f"[{c['avoidance_ci'][0]:.2f}, {c['avoidance_ci'][1]:.2f}]  gate {c['gate']}  "
              f"disguised {dis['n_disguised']}/{dis['n_banned_mechanism']}  self-names {[n for n, _ in dis['self_names_top'][:4]]}")
    for s, t in landing["tests"].items():
        if "S_edge" in t:
            ls = {f: round(v["minus_uniform"], 2) for f, v in t["log_score"].items() if isinstance(v, dict)}
            print(f"{s:<8} survivors {t['survivors']} landed {t['n_landed_survivor']}: S_edge {t['S_edge']['observed']:+.2f} "
                  f"(exact p = {t['S_edge']['p']:.2f}); log score - uniform {ls}; realised top-1 {t['realised_top1']}")
        else:
            print(f"{s:<8} survivors {t['survivors']} landed {t['n_landed_survivor']}: {t['note']}")

## Run S11: `manifold`

The stage writes the six manifold panels, the k-stability grid and the landing bars to `figures/`, and the geometry numbers to `results/manifold.json` and `results/persistence.json`. The figures are shown in the results section below.

In [ ]:
if "manifold" in STAGES:
    main(["manifold"])
    man = read_json(RESULTS_DIR / "manifold.json")["tasks"]["t1"]
    pers = read_json(RESULTS_DIR / "persistence.json")["t1"]
    print(f"\nprimary stress-1 {man['primary_stress1']:.3f} | NN stability over settings "
          f"{man['nearest_neighbour_stability_vs_primary']:.2f} | bootstrap NN kept {man['bootstrap_nearest_same']:.2f}")
    print("nearest neighbours (primary):", {SHORT[a]: [SHORT[b] for b in v] for a, v in man["primary_nearest"].items()})
    for s in man["sweep"]:
        print(f"  layer {s['layer']} {s['position']:<4} k={s['k']:<3} stress-1 {s['stress1']:.3f}  nearest "
              + ", ".join(f"{SHORT[a]}->{SHORT[b]}" for a, b in s["nearest"].items()))
    print(f"persistent H1: {pers['verdict']} (centroid-cloud p = {pers['centroid_bootstrap']['p']:.2f}, "
          f"per-sample p = {pers['per_sample']['p']:.2f}, subsample persistence {pers['per_sample']['subsample80_persistence']:.2f})")
    print("figures:", man["figures"])

## Results

1. **What the model wrote.** The demo subset's landing per condition, next to the full run (N = 100 per condition). Also shown: the full run's gate, its FRLP-implied avoidance, which was committed *before* sampling, and its disguise rate.
2. **Geometry.** The nearest neighbours and MDS stress of the algorithm centroids, demo against full run.
3. **The manifold panels** from `fig_manifold`: Isomap of the layer-24 residual states, with the programs written under each condition, centroids sized by implemented count, banned centroids crossed out, and the landing arrow.
4. **Raw activation space and centroid MDS.** A PCA of the 4096-d states of every program in the subset, and the centroid MDS of the demo (Procrustes-aligned) drawn over the full-run MDS, with the committed seriation path. The full-run centroids are sized by how often each algorithm was written across the four multi-bans.
5. **Landing bars with the committed forecasts** (`fig_landing`), the k-stability grid (`fig_kstab`), and the post-hoc check that the BFS/DFS mix is the same across the four multi-bans.

In [ ]:
%matplotlib inline
# The stage functions switch matplotlib to the file-only Agg backend; the magic above restores inline figures.
from scipy.stats import chi2_contingency

landing = read_json(RESULTS_DIR / "landing.json")
man = read_json(RESULTS_DIR / "manifold.json")["tasks"]["t1"]
ref = data["full_run_reference"]
full = ref["manifold_t1"]
algs = man["algorithms"]
MULTI = ("TOP", "ARC", "SCATTER", "TOP5")


def landing_str(counts: dict[str, int]) -> str:
    """'BFS* 5, DFS 3': * marks a banned mechanism."""
    items = sorted(counts.items(), key=lambda kv: -kv[1])
    return ", ".join(f"{SHORT.get(k.split(':', 1)[-1], k)}{'*' if k.startswith('banned:') else ''} {v}" for k, v in items)


# ---------------------------------------------------------------- 1. what the model wrote: demo subset vs full run
rows = []
for key in ["t1|baseline", "t1|single:bfs"] + [f"t1|multi:{s}" for s in MULTI]:
    d, f = landing["conditions"].get(key), ref["landing_conditions"][key]
    if d is None:
        continue
    rows.append((key.split("|")[1], ", ".join(SHORT[b] for b in f["banned"]) or "-", landing_str(d["counts"]),
                 f"{d['avoidance']:.2f}", landing_str(f["counts"]),
                 f"{f['avoidance']:.2f} [{f['avoidance_ci'][0]:.2f}, {f['avoidance_ci'][1]:.2f}]",
                 "pass" if f["gate"] else "FAIL",
                 "-" if f.get("frlp_implied_avoidance") is None else f"{f['frlp_implied_avoidance']:.3g}",
                 "-" if f["disguise_rate"] is None else f"{f['disguise_rate']:.2f}"))
hdr = ("condition", "banned", "demo landing", "demo avoid.", "full-run landing (N = 100)", "full-run avoid. [95% CI]",
       "gate", "FRLP-implied", "disguise")
widths = [max(len(str(r[i])) for r in rows + [hdr]) for i in range(len(hdr))]
print(" | ".join(h.ljust(w) for h, w in zip(hdr, widths)))
print("-+-".join("-" * w for w in widths))
for r in rows:
    print(" | ".join(str(c).ljust(w) for c, w in zip(r, widths)))
print("* = banned mechanism; avoidance = 1 - banned share; gate = avoidance >= 0.5 and >= 20 non-banned programs;\n"
      "FRLP-implied = avoidance predicted from teacher-forced likelihoods before sampling; disguise = share of "
      "banned-mechanism programs that never name their mechanism")

# ---------------------------------------------------------------- 2. geometry: demo vs full run
print(f"\nCentroid geometry (layer {PRIMARY['layer']}, '{PRIMARY['pos']}', k = {PRIMARY['k']}): MDS stress-1 "
      f"demo {man['primary_stress1']:.3f} vs full run {full['primary_stress1']:.3f}")
for a in algs:
    print(f"  {SHORT[a]:<6} nearest -> demo: {', '.join(SHORT[b] for b in man['primary_nearest'][a][:2]):<13} "
          f"full run: {', '.join(SHORT[b] for b in full['primary_nearest'][a][:2])}")

# ---------------------------------------------------------------- 3. the manifold panels (fig_manifold)
panels = ["baseline", "single_bfs"] + [f"multi_{s}" for s in MULTI]
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
for ax, p in zip(axes.ravel(), panels):
    fp = FIG_DIR / f"fig_manifold_t1_{p}.png"
    if fp.exists():
        ax.imshow(plt.imread(fp))
    ax.axis("off")
fig.suptitle("Where the written programs land on the algorithm manifold (Isomap of Qwen3-8B layer-24 residual states)\n"
             "grey: forced programs | red x: programs written under the condition | disc size/colour: implemented count | "
             "black X: banned | arrow: banned centroids -> landing", fontsize=11)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------- 4. raw activation space and centroid MDS
st_all = load_states("t1", "base")
li, pi = LAYERS.index(PRIMARY["layer"]), POS[PRIMARY["pos"]]
Y = st_all["X"][:, li, pi, :].astype(np.float32)
kind = np.array([m["kind"] for m in st_all["meta"]])
lab = np.array([m["label"] for m in st_all["meta"]])
cond = np.array([m.get("condition", "forced") for m in st_all["meta"]])
mu, V = pca_fit(Y[kind == "forced"], 2)
P = (Y - mu) @ V.T
col = dict(zip(algs, plt.get_cmap("tab10").colors))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 7))
for a in algs:
    m = (kind == "forced") & (lab == a)
    ax1.scatter(P[m, 0], P[m, 1], s=28, color=col[a], alpha=0.75, label=f"forced {SHORT[a]}")
markers = {"baseline": "o", "single:bfs": "s", "multi:TOP": "^", "multi:ARC": "v", "multi:SCATTER": "D", "multi:TOP5": "P"}
for c, mk in markers.items():
    m = (kind == "natural") & (cond == c)
    ax1.scatter(P[m, 0], P[m, 1], s=80, marker=mk, facecolors="none", linewidths=1.6,
                edgecolors=[col.get(l, (0, 0, 0)) for l in lab[m]], label=f"written: {c}")
ax1.set_xlabel("PC 1 (fitted on the forced-program states)")
ax1.set_ylabel("PC 2")
ax1.set_title(f"Raw activation space: PCA of the 4096-d states (layer {PRIMARY['layer']}, '{PRIMARY['pos']}')\n"
              "written programs: marker = condition, edge colour = mechanism label (black = other)", fontsize=10)
ax1.legend(fontsize=7, ncol=2, loc="best")

A = np.array([man["primary_mds"][a] for a in algs])
B = np.array([full["primary_mds"][a] for a in algs])
A0, B0 = A - A.mean(0), B - B.mean(0)
U, _, Wt = np.linalg.svd(A0.T @ B0)
A_al = A0 @ (U @ Wt) * (np.linalg.norm(B0) / np.linalg.norm(A0))  # orthogonal Procrustes + scale onto the full run
pooled = collections.Counter()
for s in MULTI:
    for k, v in ref["landing_conditions"][f"t1|multi:{s}"]["counts"].items():
        pooled[k.split(":", 1)[-1]] += v
mx = max(pooled.values())
cm = plt.get_cmap("viridis")
path = data["committed_geometry"]["seriation_path"]
ax2.plot([B0[algs.index(a), 0] for a in path], [B0[algs.index(a), 1] for a in path], "--", color="#999999", lw=1,
         zorder=1, label="committed Held-Karp seriation (full run)")
for i, a in enumerate(algs):
    n = pooled.get(a, 0)
    ax2.scatter(*B0[i], s=150 + 1500 * n / mx, color=cm(n / mx), edgecolor="k", zorder=3)
    ax2.plot([B0[i, 0], A_al[i, 0]], [B0[i, 1], A_al[i, 1]], color="#d62728", lw=0.8, zorder=2)
    ax2.scatter(*A_al[i], s=60, marker="x", color="#d62728", zorder=4)
    ax2.annotate(f"{SHORT[a]} ({n})", B0[i], xytext=(9, 9), textcoords="offset points", fontsize=9)
ax2.scatter([], [], s=60, marker="x", color="#d62728", label="demo centroid (56 programs, Procrustes-aligned)")
ax2.set_title(f"Centroid MDS of the geodesic distance matrix (full run: stress-1 {full['primary_stress1']:.3f})\n"
              "disc size/colour = programs of that mechanism written across the four multi-bans (full run, N = 400)",
              fontsize=10)
ax2.set_xlabel("MDS 1")
ax2.set_ylabel("MDS 2")
ax2.legend(fontsize=8, loc="best")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------- 5. landing bars, k-stability, BFS/DFS mix
display(Image(filename=str(FIG_DIR / "fig_landing.png")))
display(Image(filename=str(FIG_DIR / "fig_kstab_t1.png"), width=260))
mix = ref["posthoc_mix_homogeneity"]["bfs_dfs_counts"]
chi2, p, _, _ = chi2_contingency(np.array([mix[s] for s in MULTI]))
print("Post hoc (full run): BFS share among BFS+DFS programs per multi-ban:",
      {s: round(mix[s][0] / sum(mix[s]), 2) for s in MULTI}, f"-> chi-square {chi2:.2f}, p = {p:.2f}")

## What this shows, and how to go further

- **Bans subtract only the named target, and only one at a time.** Ban BFS alone and Qwen3-8B moves to DFS, the next rung of its likelihood ladder. Ban four or five names and it mostly goes back to BFS or DFS anyway: avoidance is 0.13, 0.37, 0.04 and 0.00 for TOP, ARC, SCATTER and TOP5. Between 41 % and 92 % of those banned-mechanism programs never name what they implement, e.g. "backtracking".
- **The landing is not explained by geometry.** The geometry itself is well organised: the centroids embed in 2-D almost perfectly (stress-1 0.025), the neighbour pairs BFS–BiBFS, DFS–IDDFS and A*–greedy are stable, and there is no loop (persistent H1). But only ARC and SCATTER put any mass on a survivor. On ARC the ladder-type forecasts (DFS) beat the geometric PROXIMITY forecast (IDA*), and on SCATTER only 2 A* programs landed. Across the four lists the BFS/DFS mix is the same (chi-square p = 0.48): a long ban list acts as a generic perturbation of the default, not as a list-specific veto.
- **What the raw activation space shows in this subset.** In the PCA panel, a group of programs written under the bans sits apart from every forced cluster, near PC 1 ≈ 40. It holds BFS- and DFS-labelled programs from the single and multi-bans and none from the baseline. None of them uses `collections.deque`: the BFS ones queue on a plain list with `pop(0)`, the DFS ones mostly use an explicit stack. All four baseline programs and all 8 forced BFS programs use `deque`. Under a ban the model keeps the mechanism but changes its idiom. This fits `pre` being the token just before the first mechanism-evidence token, which depends on how the code is written (deviation D-pre-leak).
- **The demo subset.** It is far smaller than the full run (8 programs per condition and 8 forced programs per algorithm), so its avoidance values, ARC/SCATTER re-derivation and stress are illustrations of the method. The table and figures put them next to the full-run numbers.
- **Caveats from the full study.** The `pre` position depends on the label (deviation D-pre-leak), so `g16` is the cleaner position for probes. FRLP importance weights have ESS < 5 in 97 % of cells. bf16 scoring adds about 1 nat of noise per program.
- **Scaling up.** The config cell lists the original repetition counts. The whole study's outputs (1,950 programs in `method_out.json`, `results/`, `predictions/`, `figures/`) are in the repository. The 0.95 GB of residual states and the GPU stages (`uv run method.py score1a`, `score2`, ...) are needed to rerun the full sweep over layers 20-28; see `reproducibility.md` in the repository.